# ComPartMENT 基线：Qwen2.5-1.5B-Instruct 的组合性回归管线

**任务**：给定一个多词表达（MWE，如 `account book` / `crack down`）和一句真实语料，
输出该表达**在这条语境里**的组合性分数（0–5 连续值）。

**这份 notebook 交付什么**：一套**可以直接当基线**的训练管线，主干是
`Qwen2.5-1.5B-Instruct`（`Qwen/Qwen2.5-1.5B-Instruct`）。

**跑什么**：打开 notebook 一路执行到底，它会**自动完成 5 次训练**——
四个子集各单独训练一次（`en-nn` / `de-nn` / `en-pv` / `de-pv`），
最后再把四个子集拼起来联合训练一次。
**不需要手动切换任何开关**，结果按「运行 × 评测设置」汇总成表、画成图。

**与另一条基线（DeBERTa / Qwen）的关系**：两条基线**共用同一份数据处理、
同一套超参、同一套指标、同一套绘图**，只切换 `MODEL_KEY` 一个变量。
这样后面做对比、做消融、做架构替换时，结论才能归因到「架构」这一个变量上。

---

## 五个部分

| 部分 | 对应章节 | 内容 |
|---|---|---|
| 问题与背景 | 本节 + Step1 | 三项官方指标各自的含义，指标 2 为什么是硬门槛 |
| 原理与设计 | Step3 / Step4 | 输入怎么构造、目标怎么定位、主干怎么接回归头、损失怎么写 |
| 实现代码 | Step1–Step8 | 全部可执行代码，每段都带「为什么」 |
| 结果分析 | Step9 | **评估标准**（模型在 6 个评测设置上的三项指标）与**参考对照**（随机初始化 / type-level oracle）分区呈现 |
| 结论与延伸 | 文末 | 运行前置条件、显存降级顺序、判断实验有没有做对的三个指标 |

---

## 全局数据流（谁只读、谁被更新）

```
train/{subset}-train.tsv
        │  ① 载入：删 ContextID + 物理打乱 + 去重            （Step2）
        ▼
   DataFrame
        │  ② 定位 MWE → 标记上下文 → 拼输入模板              （Step3）
        ▼
   {"text", "labels[2]", "loss_mask[2]", "group_index"}
        │  ③ 分词（动态 padding）→ 目标级互斥切分             （Step3）
        ▼
   datasets.Dataset ──Collator──► batch{input_ids, attention_mask, labels, group_index}
        │
        ▼
   AutoModel 主干（只读参数，全部更新）
        │  ④ 池化 → 线性回归头 → sigmoid×5                   （Step4）
        ▼
   logits ∈ R^{B×2}
        │  ⑤ L = Huber(掩码加权) + λ·组内排序 + μ·组内std     （Step7）
        ▼
   反向传播 → 更新主干与回归头
        │
        ▼
   评估：指标1 ρ / 指标2 ρ / 指标3 RMSE                     （Step5, Step9）
```

- **只读**：数据文件、标签列。
- **被更新**：主干全部参数 + 回归头。
- **不参与训练**：目标字符串只作为 `group_index`（分组键）与输入里的 MWE 文本，
  **不作为任何数值特征**；`ContextID` 在载入时就被删除，防止行序泄漏。

## Step1 导入包

**这一节要解决什么**：把「会随机器、随人变」的东西全部钉死在一个地方——
模型下载端点、随机种子、设备、精度、序列长度、批大小、模型 id、学习率、运行计划。
其余 8 个 Step 一律不写死任何常量，只从这里取。

**为什么必须这样**：本项目要跑 2 个模型 × 4 个子集 × 后续若干组消融。
如果 `lr=2e-5` 写在 Step6、`max_length=160` 写在 Step3，
那么「换个模型跑一遍」就要改 6 个地方，改漏一处就会得到一张**无法归因**的对照表。
对照实验的可信度 100% 取决于配置是否只有一个入口。

**先说一件本项目特有的事：模型权重从哪儿下。**

云端 GPU（Kaggle / Colab / 阿里云等海外实例）**可以直连 `huggingface.co`**，
本机直连通常不通。所以默认策略是：

1. **先探测官方源**（`https://huggingface.co`）是否真的可达——用一次真实的 HTTPS 请求，
   不是 ping，也不是 socket 探测（很多网络环境 443 端口能连但 HTTP 被重置）；
2. 官方源可达 → 用官方；不可达 → 换镜像站 `https://hf-mirror.com`；
3. **下载时再兜一层**：即便探测时通了、下载时断了，也会自动换另一个端点重试。

探测与端点切换都发生在 `import transformers` **之前**或**显式传参**，
不依赖 `HF_ENDPOINT` 全局变量的缓存时机——这是「我明明改了镜像怎么还在走官方」
这类现象最常见的来源。

In [ ]:





import os
import urllib.request





os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "expandable_segments:True")




























_GPU_INDEX = int(os.environ.get("COMPARTMENT_GPU", "0"))
os.environ["CUDA_VISIBLE_DEVICES"] = str(_GPU_INDEX)
print(f"[环境] CUDA_VISIBLE_DEVICES={os.environ['CUDA_VISIBLE_DEVICES']} "
      f"（本进程只看见这 1 张卡 → Trainer 不会套 nn.DataParallel → 梯度显存不翻倍）")

OFFICIAL_ENDPOINT = "https://huggingface.co"
MIRROR_ENDPOINT = "https://hf-mirror.com"






HF_MODE = os.environ.get("COMPARTMENT_HF_MODE", "auto").strip().lower()


def probe_http(url, timeout=8):
    try:
        req = urllib.request.Request(url, headers={"User-Agent": "compartment-probe"})
        with urllib.request.urlopen(req, timeout=timeout) as resp:
            return 200 <= resp.status < 500
    except Exception:
        return False


def choose_endpoints(model_id):
    if HF_MODE == "cache":
        print("端点策略: cache（只用本地缓存，不联网）")
        return []
    if HF_MODE == "mirror":
        print("端点策略: mirror（强制镜像站）")
        return [MIRROR_ENDPOINT, OFFICIAL_ENDPOINT]
    if HF_MODE == "official":
        print("端点策略: official（强制官方源）")
        return [OFFICIAL_ENDPOINT, MIRROR_ENDPOINT]
    api = f"{OFFICIAL_ENDPOINT}/api/models/{model_id}"
    ok = probe_http(api)
    print(f"官方源连通性探测: {'通' if ok else '不通'}  ({api})")
    if ok:
        print("→ 使用官方 huggingface.co（云端 GPU 直连即走这条）")
        return [OFFICIAL_ENDPOINT, MIRROR_ENDPOINT]
    print("→ 官方源不可达，改用镜像站 hf-mirror.com")
    return [MIRROR_ENDPOINT, OFFICIAL_ENDPOINT]


os.environ.setdefault("TOKENIZERS_PARALLELISM", "false")

In [ ]:

import gc
import inspect
import json
import math
import platform
import re
import shutil
import time
import unicodedata
import warnings
from collections import Counter
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from scipy import stats
from scipy.stats import ConstantInputWarning

import matplotlib.pyplot as plt
from matplotlib import font_manager

from datasets import Dataset
from transformers import (
    AutoConfig,
    AutoModel,
    AutoTokenizer,
    Trainer,
    TrainingArguments,
    set_seed,
)
from transformers.modeling_outputs import ModelOutput
from torch.utils.data import Sampler

print("torch", torch.__version__)

In [ ]:










import tempfile
from urllib.request import Request as _UrlRequest, urlopen as _urlopen

_FONT_CANDIDATES = [
    "Arial Unicode MS",
    "Noto Sans CJK SC",
    "Noto Sans SC",
    "Source Han Sans SC",
    "WenQuanYi Zen Hei",
    "Hiragino Sans GB",
    "Songti SC",
    "STHeiti",
    "PingFang HK",
    "PingFang SC",
    "SimHei",
    "DejaVu Sans",
]
_CJK_FONT_URLS = [
    "https://github.com/googlefonts/noto-cjk/raw/main/Sans/OTF/SimplifiedChinese/NotoSansCJKsc-Regular.otf",
    "https://cdn.jsdelivr.net/gh/googlefonts/noto-cjk@main/Sans/OTF/SimplifiedChinese/NotoSansCJKsc-Regular.otf",
    "https://mirrors.tuna.tsinghua.edu.cn/CTAN/fonts/noto-cjk/NotoSansCJKsc-Regular.otf",
]


def _find_installed_cjk():
    try:
        installed = {f.name for f in font_manager.fontManager.ttflist}
    except Exception:
        installed = set()
    for name in _FONT_CANDIDATES:
        if name in installed:
            return name
    return None


def _download_cjk_font():
    target_dir = Path(tempfile.gettempdir()) / "compartment_fonts"
    target_dir.mkdir(parents=True, exist_ok=True)
    target = target_dir / "NotoSansCJKsc-Regular.otf"
    if not target.exists():
        for url in _CJK_FONT_URLS:
            try:
                req = _UrlRequest(url, headers={"User-Agent": "compartment"})
                with _urlopen(req, timeout=90) as resp:
                    target.write_bytes(resp.read())
                break
            except Exception:
                continue
    if not target.exists() or target.stat().st_size < 1_000_000:
        return None
    try:
        font_manager.fontManager.addfont(str(target))
        return font_manager.FontProperties(fname=str(target)).get_name()
    except Exception:
        return None


_CHOSEN_FONT = None
if os.environ.get("COMPARTMENT_NO_CJK", "") == "1":

    print("COMPARTMENT_NO_CJK=1 → 强制使用英文图注")
else:
    _CHOSEN_FONT = _find_installed_cjk()
    if _CHOSEN_FONT is None:
        _CHOSEN_FONT = _download_cjk_font()
_HAS_CJK = _CHOSEN_FONT is not None
if not _HAS_CJK:
    _CHOSEN_FONT = "DejaVu Sans"
    print("⚠️ 找不到中文字体且下载失败 → 图上文字自动切换为英文（PLOT_LANG='en'）")
else:
    print("中文字体:", _CHOSEN_FONT)

plt.rcParams["font.sans-serif"] = [_CHOSEN_FONT] + [f for f in _FONT_CANDIDATES if f != _CHOSEN_FONT]
plt.rcParams["axes.unicode_minus"] = False


PLOT_LANG = "zh" if _HAS_CJK else "en"

_L10N = {
    "zh": {
        "gold": "人工评分", "pred": "模型预测",
        "gold_std": "人工评分的组内标准差（金标准）", "pred_std": "模型预测的组内标准差",
        "m1": "指标1_ρ", "m2": "指标2_ρ", "m3": "指标3_RMSE",
        "rho_up": "Spearman ρ（越大越好）", "rmse_down": "RMSE（越小越好）",
        "figA": "图A  预测 vs 真值", "figB": "图B  评估标准", "figC": "图C  指标2 的可视化",
        "figD": "图D  五次运行的合计指标对照", "figE": "图E  参考对照",
        "not_standard": "非评估标准", "trained": "训练后模型", "random_init": "随机初始化",
        "oracle": "type-level oracle", "single": "单子集训练", "joint": "联合训练（覆盖全部设置）",
        "joint_short": "联合\n(四子集)", "single_short": "单子集\n{short}",
        "m2rho": "指标2 ρ =", "n_targets": "目标数 =",
        "const_std": "预测 std 恒为 0 的目标数 =", "n_settings": "个评测设置",
        "gold_axis": "横轴金标准、纵轴系统输出", "eval_std": "评估标准",
        "short": {},
    },
    "en": {
        "gold": "Gold score", "pred": "Prediction",
        "gold_std": "Gold within-target std", "pred_std": "Predicted within-target std",
        "m1": "Metric1 rho", "m2": "Metric2 rho", "m3": "Metric3 RMSE",
        "rho_up": "Spearman rho (higher is better)", "rmse_down": "RMSE (lower is better)",
        "figA": "Fig A  Prediction vs Gold", "figB": "Fig B  Evaluation standard",
        "figC": "Fig C  Metric 2", "figD": "Fig D  Aggregate metrics across runs",
        "figE": "Fig E  Reference systems",
        "not_standard": "reference only", "trained": "Trained", "random_init": "Random init",
        "oracle": "type-level oracle", "single": "Single-subset", "joint": "Joint (all settings)",
        "joint_short": "Joint\n(4 subsets)", "single_short": "single\n{short}",
        "m2rho": "Metric2 rho =", "n_targets": "targets =",
        "const_std": "targets with constant pred std =", "n_settings": "settings",
        "gold_axis": "gold (x) vs system (y)", "eval_std": "Evaluation standard",
        "short": {"英·复合词": "EN-NN", "德·复合词": "DE-NN",
                  "英·小品词": "EN-PV", "德·小品词": "DE-PV"},
    },
}


def T(key, **kw):
    v = _L10N[PLOT_LANG].get(key, key)
    return v.format(**kw) if kw else v


def short_of(subset):
    zh = SUBSET_SPECS[subset]["short"]
    return _L10N[PLOT_LANG]["short"].get(zh, zh)

In [ ]:

SEED = 42


MODEL_REGISTRY = {
    "deberta": dict(
        model_id="microsoft/deberta-v2-xlarge",
        pooling="mean",
        attn_impls=("eager",),
    ),
    "qwen": dict(
        model_id="Qwen/Qwen2.5-1.5B-Instruct",
        pooling="last",
        attn_impls=("sdpa", "eager"),
    ),
}



MODEL_KEY = "qwen"






RUN_PLAN = [
    ("single", "en-nn"),
    ("single", "de-nn"),
    ("single", "en-pv"),
    ("single", "de-pv"),
    ("joint", None),
]


SUBSET_SPECS = {
    "en-nn": dict(key="Compound",     cols=["ModAvg", "HeadAvg"], lang="en", task="nn", short="英·复合词"),
    "de-nn": dict(key="Compound",     cols=["ModAvg", "HeadAvg"], lang="de", task="nn", short="德·复合词"),
    "en-pv": dict(key="ParticleVerb", cols=["Avg"],             lang="en", task="pv", short="英·小品词"),
    "de-pv": dict(key="ParticleVerb", cols=["Avg"],             lang="de", task="pv", short="德·小品词"),
}
SUBSET_ORDER = ["en-nn", "de-nn", "en-pv", "de-pv"]





N_OUTPUTS = 2


MAX_LENGTH   = 160


TARGET_EFFECTIVE_BATCH = 8
BATCH_SIZE   = 8
GRAD_ACCUM   = 1
PROBE_MEMORY = True
LR           = 2e-5
EPOCHS       = 3
VAL_RATIO    = 0.2
WEIGHT_DECAY = 0.01
WARMUP_RATIO = 0.1
HUBER_DELTA  = 1.0






















USE_8BIT_OPT   = "auto"
USE_GRAD_CKPT  = False













FIT_MODE       = "auto"
FIT_REF_PARAMS = 1.55e9
FIT_SAFETY     = 0.90







GPU_INDEX      = int(os.environ.get("COMPARTMENT_GPU", "0"))


TIME_BUDGET_H = 11.0


LAMBDA_RANK = 0.0
MU_STD      = 0.0


GROUPED_BATCHING = True
SMOKE = False
SMOKE_N_ROWS = 400





SAVE_FINAL_FOR = {"joint"}





if SMOKE:
    MAX_LENGTH = 64
    EPOCHS = 1
    BATCH_SIZE = 4






    RUN_PLAN = [("single", "en-nn"), ("single", "de-nn"), ("joint", None)]

print(json.dumps(dict(MODEL_KEY=MODEL_KEY, MAX_LENGTH=MAX_LENGTH, BATCH_SIZE=BATCH_SIZE,
                      LR=LR, EPOCHS=EPOCHS, LAMBDA_RANK=LAMBDA_RANK, MU_STD=MU_STD,
                      SMOKE=SMOKE, RUN_PLAN=[list(x) for x in RUN_PLAN]),
                 ensure_ascii=False, indent=1))

In [ ]:

def detect_device():
    if torch.cuda.is_available():
        _n_gpu = torch.cuda.device_count()
        _gi = GPU_INDEX % max(1, _n_gpu)




        torch.cuda.set_device(_gi)
        _cap = torch.cuda.get_device_capability(_gi)
        _native_bf16 = (_cap[0] >= 8) and torch.cuda.is_bf16_supported()
        _fp16 = (not _native_bf16) and (_cap[0] >= 7)
        return "cuda", _native_bf16, _fp16, True
    if getattr(torch.backends, "mps", None) is not None and torch.backends.mps.is_available():

        return "mps", False, False, False
    return "cpu", False, False, False


DEVICE, USE_BF16, USE_FP16, USE_TF32 = detect_device()


HW_BF16, HW_FP16 = USE_BF16, USE_FP16




NUM_WORKERS = 0 if platform.system() == "Darwin" else max(1, (os.cpu_count() or 2) // 2)
DATALOADER_PERSISTENT = NUM_WORKERS > 0


if USE_TF32:
    torch.backends.cuda.matmul.allow_tf32 = True
    torch.backends.cudnn.allow_tf32 = True

set_seed(SEED)
print({
    "device": DEVICE, "bf16": USE_BF16, "fp16": USE_FP16, "tf32": USE_TF32,
    "workers": NUM_WORKERS, "torch": torch.__version__, "platform": platform.system(),
    "python": platform.python_version(),
})



if torch.cuda.is_available():
    for _i in range(torch.cuda.device_count()):
        _p = torch.cuda.get_device_properties(_i)
        print(f"GPU{_i}: {_p.name} | {_p.total_memory / 1024 ** 3:.1f} GiB | "
              f"compute capability {_p.major}.{_p.minor} | "
              f"原生bf16={_p.major >= 8} | fp16={_p.major >= 7}")
    _total_cards = sum(torch.cuda.get_device_properties(_i).total_memory
                        for _i in range(torch.cuda.device_count())) / (1024 ** 3)
    print(f"物理卡：{torch.cuda.device_count()} 张可见（合计 {_total_cards:.1f} GiB）；"
          f"本 notebook 绑定物理 GPU{GPU_INDEX}，在进程内编号为 cuda:0")
    print("→ 单卡模式：Trainer 看到 n_gpu=1 → **不会**自动套 nn.DataParallel"
          "（DataParallel 的反向传播会额外复制一整份梯度，Qwen-1.5B 要多花 5.75 GiB）")

## Step2 加载数据集

**这一节要解决什么**：用最少的假设把原始 TSV 变成规整的 DataFrame，
并且**在载入这一步就把会泄漏答案的东西扔掉**。

**为什么载入阶段就要处理反泄漏**：本项目实测发现，
同一个 MWE 的各条例句在文件里是**按 ContextID 顺序连续排列**的，
而 ContextID 又与该目标的平均分高度相关（组内 Spearman 达 0.985）。
换句话说，**「这条例句在文件里的位置」本身就带着答案**。
如果先建特征再打乱，任何从行号派生的东西（下标、位次、窗口邻域）都会变成作弊通道。
所以顺序必须是：**先删 ContextID → 再物理打乱**。

**要保留什么**：`Compound` / `ParticleVerb`（MWE 本身）、`Mod`/`Head`/`Base`/`Particle`（成分）、
`ModAvg`/`HeadAvg`/`Avg`（标签）、`Context`（句子）。

In [ ]:






CLOUD_TRAIN_DIR = "/kaggle/input/datasets/kintsugi0v0/compartment-data/train"


def resolve_data_root(start: Path) -> Path:
    cands = []
    env = os.environ.get("COMPARTMENT_TRAIN_DIR")
    if env:
        cands.append(Path(env).expanduser())
    cands.append(Path(CLOUD_TRAIN_DIR))

    root_env = os.environ.get("COMPARTMENT_ROOT")
    if root_env:
        cands.append(Path(root_env).expanduser() / "data" / "train")
        cands.append(Path(root_env).expanduser() / "train")
    p = start.resolve()
    for cand in [p] + list(p.parents):
        for rel in ("data/train", "train"):
            if (cand / rel / "en-nn-train.tsv").exists():
                cands.append(cand / rel)
                break
        else:
            continue
        break
    for c in cands:
        if (c / "en-nn-train.tsv").exists():
            return c
    raise FileNotFoundError(
        "找不到 en-nn-train.tsv。已依次尝试：\n  " + "\n  ".join(str(c) for c in cands)
        + "\n请设置环境变量 COMPARTMENT_TRAIN_DIR 指向含 4 个 *-train.tsv 的目录。")


WORK_DIR = Path.cwd()
DATA_ROOT = resolve_data_root(WORK_DIR)

OUTPUT_ROOT = Path(os.environ.get("COMPARTMENT_OUTPUT", str(WORK_DIR / "outputs")))
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
print("WORK_DIR  =", WORK_DIR)
print("DATA_ROOT =", DATA_ROOT)
print("OUTPUT_ROOT =", OUTPUT_ROOT)

In [ ]:



_demo_name = SUBSET_ORDER[0]
_demo_spec = SUBSET_SPECS[_demo_name]
raw = pd.read_csv(DATA_ROOT / f"{_demo_name}-train.tsv", sep="\t")
print(f"{_demo_name} 原始形状: {raw.shape}")
print("列名:", list(raw.columns))
print(raw[[_demo_spec["key"]] + _demo_spec["cols"] + ["Context"]].head(3).to_string()[:700])


print("\n--- 标签分布 ---")
for c in _demo_spec["cols"]:
    s = raw[c]
    print(f"{c}: 均值 {s.mean():.3f} | 标准差 {s.std():.3f} | 最小 {s.min():.2f} | 最大 {s.max():.2f}")
wlen = raw["Context"].str.split().str.len()
print("\n--- 句子长度（词）分位 ---")
print(dict(zip(["p50", "p90", "p95", "p99", "max"],
                np.percentile(wlen, [50, 90, 95, 99, 100]).round(0).tolist())))
print("→ 句子很短，MAX_LENGTH=160 足够，不会因为截断丢掉关键信息。")

In [ ]:

def load_subset(name, seed=SEED):
    sp = SUBSET_SPECS[name]
    df = pd.read_csv(DATA_ROOT / f"{name}-train.tsv", sep="\t")
    df = df.drop_duplicates(subset=[sp["key"], "Context"]).reset_index(drop=True)
    if "ContextID" in df.columns:
        df = df.drop(columns=["ContextID"])
    df = df.sample(frac=1, random_state=seed).reset_index(drop=True)
    df["y0"] = df[sp["cols"][0]].astype(float)
    df["m0"] = 1.0
    if len(sp["cols"]) > 1:
        df["y1"] = df[sp["cols"][1]].astype(float)
        df["m1"] = 1.0
    else:
        df["y1"] = 0.0
        df["m1"] = 0.0
    if SMOKE and SMOKE_N_ROWS:



        keep = df[sp["key"]].drop_duplicates().head(max(1, SMOKE_N_ROWS // 5))
        df = df[df[sp["key"]].isin(set(keep))].reset_index(drop=True)
    return df


for _n in SUBSET_ORDER:
    _d = load_subset(_n)
    _sp = SUBSET_SPECS[_n]
    print(f"[{_n}] 形状 {_d.shape} | 目标数 {_d[_sp['key']].nunique()} | "
          f"每目标例句数中位 {_d.groupby(_sp['key']).size().median()} | "
          f"ContextID 是否还在: {'ContextID' in _d.columns}")

## Step3 数据预处理

这一节做四件事，每件都有明确的动机：

| 环节 | 做什么 | 为什么 |
|---|---|---|
| ① 定位 MWE | 在句子里标出 MWE 出现的那个 span | 任务问的是「这个 MWE 在**这句**里是不是被当成整体理解」。不告诉模型句中哪个 span 对应 MWE，它只能靠整句语义猜 |
| ② 拼输入模板 | `MWE: <mwe> \| Context: <ctx>` | 两个模型必须看到**逐字相同**的输入，否则「模型差异」里会混进「模板差异」 |
| ③ 目标级互斥切分 | 按 MWE 切 train/val，并复刻官方「先分箱后分层抽样」 | 按行切会把同一 MWE 的例句分到两边，模型等于「见过这个 MWE」再去预测它，指标被高估 |
| ④ 分词 + 动态 padding | 批量分词，batch 内补齐到 8 的倍数 | 定长补齐到 `max_length` 会把每条的注意力开销抬到 O(L²) 的上限，动态补齐只按本批最长序列付费 |

**定位为什么难**：本项目实测过定位率——
英语复合词 98.6%、德语复合词 95.9% 可以字面命中；
但英语小品词动词只有 33.6%（`crack down` 在句子里常写成 `cracking down`），
德语更只有 20.8%（德语可分动词写作 `abgehauen`，粒子粘在动词前，且中间插前缀 `ge-`）。
所以定位必须做**分级级联**，而不是一次精确匹配。

In [ ]:






def fold(s, ascii_mode=False):
    s = unicodedata.normalize("NFKD", str(s).lower()).replace("ß", "ss")
    s = re.sub(r"[^a-z0-9äöüß']", "", s)
    if ascii_mode:
        for a, b in [("ä", "ae"), ("ö", "oe"), ("ü", "ue")]:
            s = s.replace(a, b)
    return s


def stem_de(b):
    for suf in ["ern", "eln", "en", "er", "e", "t", "ten", "n"]:
        if b.endswith(suf) and len(b) - len(suf) >= 4:
            return b[:-len(suf)]
    return b


def stem_en(w):
    for suf in ["ing", "edly", "ed", "es", "s", "y", "e"]:
        if w.endswith(suf) and len(w) - len(suf) >= 3:
            return w[:-len(suf)]
    return w


def stem_keys(base, lang):
    ks = set()
    for v in (fold(base), fold(base, True)):
        st = stem_de(v) if lang == "de" else stem_en(v)
        for L in (5, 4, 3):
            if len(st) >= L:
                ks.add(st[:L])
    return ks


WORD_RE = re.compile(r"[a-zA-ZäöüÄÖÜß]+(?:[-'][a-zA-ZäöüÄÖÜß]+)*")
print("归一化与词干工具就绪")

In [ ]:




_demo_df = load_subset(_demo_name)
_sample_row = _demo_df.iloc[0].to_dict()
_mwe = _sample_row[_demo_spec["key"]]
_context = _sample_row["Context"]
print("MWE   :", _mwe)
print("原句  :", _context[:170])


_toks = [(m.group(0), m.start(), m.end()) for m in WORD_RE.finditer(_context.lower())]
print(f"\n句子切成 {len(_toks)} 个词:", [t[0] for t in _toks][:14])


print("\n归一化与词干:")
print("  fold(MWE) =", fold(_mwe))
if _demo_spec["task"] == "nn":
    print("  词干键(复合词) =", sorted(stem_keys(_mwe, _demo_spec["lang"])))
else:
    print("  fold(基干)   =", fold(_sample_row["Base"]))
    print("  词干键(基干) =", sorted(stem_keys(_sample_row["Base"], _demo_spec["lang"])))
    print("  粒子         =", _sample_row["Particle"])


_m = re.search(re.escape(_mwe.lower()).replace(r"\-", r"[-']?").replace(r"\ ", r"[-']?\s"),
               _context.lower())
print("\n最小版本（只做 S0 精确匹配）:", ("命中 → " + repr(_context[_m.start():_m.end()])) if _m else "未命中")
print("→ 精确匹配在复合词上够用，但在小品词动词上只能覆盖约三分之一；")
print("   所以下一格把级联封装成一个函数，逐级放宽。")

In [ ]:

def _mark(text, start, end):
    return text[:start] + "[[" + text[start:end] + "]]" + text[end:]


def locate_target(row, sp):
    ctx = str(row["Context"])
    low = ctx.lower()
    mwe = str(row[sp["key"]])
    if sp["task"] == "nn":
        base, part = mwe, ""
    else:
        base, part = str(row["Base"]), str(row["Particle"])


    m = re.search(re.escape(mwe.lower()).replace(r"\-", r"[-']?").replace(r"\ ", r"[-']?\s"), low)
    if m:
        return _mark(ctx, m.start(), m.end()), "S0 精确匹配"

    toks = [(mm.group(0), mm.start(), mm.end()) for mm in WORD_RE.finditer(low)]
    if not toks:
        return ctx, "S5 未定位"


    if fold(mwe, True) and fold(mwe, True) in fold(low, True):
        return ctx, "S1 历史转写还原"

    ks = stem_keys(base, sp["lang"])
    p = fold(part)
    pw = {fold(part), fold(part, True)}
    fw = [fold(t[0]) for t in toks]
    fa = [fold(t[0], True) for t in toks]

    if sp["lang"] == "de":

        for i2, t in enumerate(toks):
            if not fw[i2].startswith(p):
                continue
            r, ra = fw[i2][len(p):], fa[i2][len(p):]
            if any(k and (k in r or k in ra) for k in ks):
                return _mark(ctx, t[1], t[2]), "S2a 德语前缀型(屈折)"

        for i2 in range(len(toks)):
            if not any(k and (k in fw[i2] or k in fa[i2]) for k in ks):
                continue
            for j2 in range(i2 + 1, min(i2 + 4, len(toks))):
                if fw[j2] in pw or fa[j2] in pw:
                    return _mark(ctx, toks[i2][1], toks[j2][2]), "S2b 德语后置型(动词分离)"
        return ctx, "S5 未定位"


    for i2, t in enumerate(toks):
        if any(k and (k in fw[i2] or k in fa[i2]) for k in ks):
            return _mark(ctx, t[1], t[2]), "S3 英文词干前缀"

    for i2 in range(len(toks) - 1):
        two = fw[i2] + fw[i2 + 1]
        if any(k and k in two for k in ks):
            return _mark(ctx, toks[i2][1], toks[i2 + 1][2]), "S4 相邻窗口"
    return ctx, "S5 未定位"


def build_input(mwe, marked_context):
    return f"MWE: {mwe} | Context: {marked_context}"



_marked, _strat = locate_target(_sample_row, _demo_spec)
print("策略  :", _strat)
print("标记后:", _marked[:200])
print("最终输入:", build_input(_mwe, _marked)[:220])

In [ ]:





for _n in SUBSET_ORDER:
    _sp = SUBSET_SPECS[_n]
    _d = load_subset(_n)
    _cnt = Counter()
    for _row in _d.to_dict("records"):
        _, _st = locate_target(_row, _sp)
        _cnt[_st] += 1
    _total = sum(_cnt.values())
    _loc = _total - _cnt.get("S5 未定位", 0) - _cnt.get("S1 历史转写还原", 0)
    print(f"{_n} 定位覆盖（n={_total}）成功标出 span {_loc / _total:.1%} | " +
          " | ".join(f"{k} {v}" for k, v in _cnt.most_common()))

In [ ]:

def split_by_target(df, sp, val_ratio=VAL_RATIO, seed=SEED, n_bins=5):
    key = sp["key"]
    tgt_mean = df.groupby(key)[sp["cols"]].mean().mean(axis=1)
    try:
        bins = pd.qcut(tgt_mean, n_bins, labels=False, duplicates="drop").to_numpy()
    except ValueError:
        bins = np.zeros(len(tgt_mean), dtype=int)

    rng = np.random.default_rng(seed)
    val_targets = []
    for b in np.unique(bins):
        idx = tgt_mean.index[bins == b].to_numpy()
        rng.shuffle(idx)
        k = max(1, int(round(len(idx) * val_ratio)))
        val_targets.extend(idx[:k].tolist())

    is_val = df[key].isin(set(val_targets)).to_numpy()
    return df[~is_val].reset_index(drop=True), df[is_val].reset_index(drop=True)

In [ ]:




_ALLOW_PATTERNS = ["*.json", "*.txt", "*.model", "*.safetensors", "*.bin", "*.py", "*.tiktoken"]


def resolve_model_dir(repo_id):
    from huggingface_hub import snapshot_download
    endpoints = choose_endpoints(repo_id)
    if not endpoints:
        return repo_id
    last_err = None
    for ep in endpoints:
        try:
            path = snapshot_download(repo_id, endpoint=ep, allow_patterns=_ALLOW_PATTERNS)
            print(f"权重就绪: {repo_id} @ {ep}\n         -> {path}")
            return path
        except Exception as e:
            last_err = e
            print(f"从 {ep} 下载失败: {type(e).__name__}: {str(e)[:180]}")
    raise last_err



if SMOKE:
    _SMOKE_IDS = {"deberta": "hf-internal-testing/tiny-random-DebertaV2Model",
                  "qwen": "Qwen/Qwen2.5-0.5B-Instruct"}
    MODEL_SOURCE = _SMOKE_IDS[MODEL_KEY]
    print("SMOKE=True：使用微型随机主干，不下载权重 ->", MODEL_SOURCE)
else:
    MODEL_SOURCE = resolve_model_dir(MODEL_REGISTRY[MODEL_KEY]["model_id"])

In [ ]:

def load_tokenizer(model_key, source):
    tok = AutoTokenizer.from_pretrained(source)
    if tok.pad_token is None:
        tok.pad_token = tok.eos_token
    tok.padding_side = "right"
    tok.model_max_length = 512
    return tok


def build_dataset(frame, tokenizer, max_length=MAX_LENGTH):
    records = []
    for row in frame.to_dict("records"):
        sp = SUBSET_SPECS[row["subset_name"]]
        mwe = str(row[sp["key"]])
        marked, _ = locate_target(row, sp)
        records.append({
            "text": build_input(mwe, marked),
            "labels": [float(row["y0"]), float(row["y1"])],
            "loss_mask": [float(row["m0"]), float(row["m1"])],
            "group_index": 0,
            "group_key": f'{row["subset_name"]}||{mwe}',
            "subset_index": int(row["subset_id"]),
        })
    ds = Dataset.from_list(records)


    codes = {k: i for i, k in enumerate(sorted(set(frame["group_key"].astype(str))))}
    ds = ds.map(lambda b: {"group_index": [codes[k] for k in b["group_key"]]}, batched=True)
    ds = ds.remove_columns(["group_key"])


    ds = ds.map(lambda b: tokenizer(b["text"], truncation=True, max_length=max_length),
                batched=True, remove_columns=["text"])
    return ds


def ds_labels(ds, n_out=None):
    n_out = n_out if n_out is not None else N_OUTPUTS
    arr = np.asarray(ds["labels"], dtype=float)
    if arr.ndim == 1:
        arr = arr.reshape(-1, n_out)
    return arr


tokenizer = load_tokenizer(MODEL_KEY, MODEL_SOURCE)
print("tokenizer:", type(tokenizer).__name__, "| pad:", tokenizer.pad_token,
      "| padding_side:", tokenizer.padding_side)

In [ ]:





def prepare_run(subset_list, tokenizer):
    frames_tr, frames_va = [], []
    for name in subset_list:
        sp = SUBSET_SPECS[name]
        df = load_subset(name)
        df["subset_name"] = name
        df["subset_id"] = SUBSET_ORDER.index(name)
        df["group_key"] = name + "||" + df[sp["key"]].astype(str)
        tr, va = split_by_target(df, sp)
        frames_tr.append(tr)
        frames_va.append(va)
    train_df = pd.concat(frames_tr, ignore_index=True)
    val_df = pd.concat(frames_va, ignore_index=True)
    train_ds = build_dataset(train_df, tokenizer)
    val_ds = build_dataset(val_df, tokenizer)

    eval_cols = [(n, c, j) for n in subset_list for j, c in enumerate(SUBSET_SPECS[n]["cols"])]
    return dict(subset_list=list(subset_list), train_df=train_df, val_df=val_df,
                train_ds=train_ds, val_ds=val_ds, eval_cols=eval_cols)



_demo_run = prepare_run(["en-nn"], tokenizer)
_tr, _va = _demo_run["train_df"], _demo_run["val_df"]
print(f"en-nn 训练集 {_tr.shape} | 验证集 {_va.shape}")
print("目标交集:", len(set(_tr["group_key"]) & set(_va["group_key"])), "（必须为 0）")
for _n, _c, _j in _demo_run["eval_cols"]:
    _col = "y0" if _j == 0 else "y1"
    print(f"  {_n}·{_c}: train 均值 {_tr[_col].mean():.3f} | val 均值 {_va[_col].mean():.3f}")
print("示例解码:", tokenizer.decode(_demo_run["train_ds"][0]["input_ids"])[:220])

In [ ]:




_lens = [len(x) for x in _demo_run["train_ds"]["input_ids"]]
_fixed_total = len(_lens) * MAX_LENGTH
_dynamic_total = 0
for _i in range(0, len(_lens), BATCH_SIZE):
    _dynamic_total += min(BATCH_SIZE, len(_lens) - _i) * max(_lens[_i:_i + BATCH_SIZE])
print(f"序列长度: 中位 {int(np.median(_lens))} | 均值 {int(np.mean(_lens))} | 最长 {max(_lens)}")
print(f"定长补齐到 {MAX_LENGTH} 的 token 总量: {_fixed_total:,}")
print(f"动态补齐的 token 总量            : {_dynamic_total:,}")
print(f"→ 动态补齐省下 {(1 - _dynamic_total / _fixed_total):.1%} 的 token，"
      f"注意力开销按平方计，实际省得更多。")

In [ ]:

class CompartmentCollator:
    def __init__(self, tokenizer, pad_to_multiple_of=8):
        self.tokenizer = tokenizer
        self.pad_to_multiple_of = pad_to_multiple_of

    def __call__(self, features):
        model_inputs = [{k: v for k, v in f.items()
                         if k not in ("labels", "loss_mask", "group_index", "subset_index")}
                        for f in features]
        batch = self.tokenizer.pad(model_inputs, padding=True,
                                   pad_to_multiple_of=self.pad_to_multiple_of,
                                   return_tensors="pt")
        batch["labels"] = torch.tensor([f["labels"] for f in features], dtype=torch.float32)


        batch["loss_mask"] = torch.tensor([f["loss_mask"] for f in features], dtype=torch.float32)
        batch["group_index"] = torch.tensor([f["group_index"] for f in features], dtype=torch.long)
        batch["subset_index"] = torch.tensor([f["subset_index"] for f in features], dtype=torch.long)
        return batch


collator = CompartmentCollator(tokenizer)
_demo_batch = collator([_demo_run["train_ds"][i] for i in range(4)])
for k, v in _demo_batch.items():
    print(f"{k:16s} shape={tuple(v.shape)} dtype={v.dtype}")
print("\nlabels 样例   :", _demo_batch["labels"].tolist())
print("loss_mask 样例:", _demo_batch["loss_mask"].tolist())
print("subset_index  :", _demo_batch["subset_index"].tolist())
print("→ labels 是 [B, 2] 的 float 张量，没有 -100 填充；loss_mask 标出哪一列真有标签。")

## Step4 创建模型

**这一节要解决什么**：把「编码器」和「解码器」两个完全不同的架构，
装配成**同一个形状的模型**：`(input_ids, attention_mask) → 两个 0–5 的连续分数`。

**为什么不用 `AutoModelForSequenceClassification`**：两条理由，都很硬。

1. **任务类型不匹配**。我们的标签是 0–5 的**连续均值**
   （本项目实测：每个标签是 10 个 0–5 整数评分的均值）。
   强行用 `num_labels=6` + 交叉熵，会把「4.2 分」和「4.0 分」的差别完全抹平，
   而指标 1（组合性相关）恰恰只看**排序**——抹平排序等于直接放弃指标 1。
2. **接口不统一**。DeBERTa 的 `DebertaV2ForSequenceClassification` 与
   Qwen 的 `Qwen2ForSequenceClassification` 在 pooling、损失、logits 形状上都不一致。
   自己写一个 20 行的头，两个模型才能**逐行同构**，
   对照实验里唯一的变量才真的是「架构」而不是「框架细节」。

**结构**：

```
AutoModel 主干 ──► last_hidden_state [B, L, H]
                          │
              掩码均值池化(编码器) / 取最后一个真实 token(解码器)
                          │
                       [B, H]
                          │
                  Dropout → Linear(H, n_out)
                          │
                    logits [B, n_out]
                          │
                  × 5 并落在 (0,5)
```

**为什么最后要 ×5**：标签的量纲是 0–5，模型是无约束的实数输出。
用 `sigmoid × 5` 把输出**先验地限制在合法区间**，
这样即使训练早期发散，预测值也不会跑到 [-100, 100] 这种让 RMSE 直接爆掉的范围。

In [ ]:

def shrink_config(cfg, num_layers=2, hidden_size=64, num_heads=4, tokenizer=None):
    cfg.num_hidden_layers = num_layers
    cfg.hidden_size = hidden_size
    cfg.num_attention_heads = num_heads
    cfg.intermediate_size = hidden_size * 2

    if hasattr(cfg, "attention_head_size"):
        cfg.attention_head_size = hidden_size // num_heads
    if hasattr(cfg, "head_dim"):
        cfg.head_dim = hidden_size // num_heads
    if hasattr(cfg, "embedding_size"):
        cfg.embedding_size = hidden_size
    if hasattr(cfg, "num_key_value_heads"):
        cfg.num_key_value_heads = min(int(getattr(cfg, "num_key_value_heads", 1)), num_heads)
    if tokenizer is not None:


        cfg.vocab_size = max(cfg.vocab_size, len(tokenizer))
    return cfg






_demo_cfg = shrink_config(AutoConfig.from_pretrained(MODEL_SOURCE), tokenizer=tokenizer)
_demo_backbone = AutoModel.from_config(_demo_cfg)
_demo_backbone.eval()

_demo_ids = tokenizer("MWE: account book | Context: He read his [[account book]] last night.",
                      return_tensors="pt")
with torch.no_grad():
    _demo_out = _demo_backbone(input_ids=_demo_ids["input_ids"],
                               attention_mask=_demo_ids["attention_mask"])
_h = _demo_out.last_hidden_state
print("主干类型:", type(_demo_backbone).__name__)
print("config: hidden_size=%d num_attention_heads=%d attention_head_size=%s head_dim=%s" % (
    _demo_cfg.hidden_size, _demo_cfg.num_attention_heads,
    getattr(_demo_cfg, "attention_head_size", "-"), getattr(_demo_cfg, "head_dim", "-")))
print("last_hidden_state shape:", tuple(_h.shape))


assert _h.shape[-1] == _demo_cfg.hidden_size, (
    f"主干输出宽度 {_h.shape[-1]} ≠ hidden_size {_demo_cfg.hidden_size}；"
    "多半是 config 里有派生宽度字段（attention_head_size / head_dim / embedding_size）没同步")
print("→ [batch, 序列长度, 隐层维度]。真正的「句子向量」还需要池化，这就是下面要做的。")

In [ ]:

class CompositionalityRegressor(nn.Module):

    def __init__(self, backbone, hidden_size, n_outputs=1, pooling="mean", dropout=0.1):
        super().__init__()
        self.backbone = backbone
        self.pooling = pooling
        self.dropout = nn.Dropout(dropout)
        self.head = nn.Linear(hidden_size, n_outputs)








    supports_gradient_checkpointing = True

    def gradient_checkpointing_enable(self, gradient_checkpointing_kwargs=None, **kwargs):
        if not hasattr(self.backbone, "gradient_checkpointing_enable"):
            raise RuntimeError("当前主干不支持 gradient checkpointing，无法开启")
        self.backbone.gradient_checkpointing_enable(
            gradient_checkpointing_kwargs=gradient_checkpointing_kwargs, **kwargs)

    @property
    def is_gradient_checkpointing(self):
        return bool(getattr(self.backbone, "is_gradient_checkpointing", False))

    def forward(self, input_ids=None, attention_mask=None, labels=None,
                group_index=None, **kwargs):
        out = self.backbone(input_ids=input_ids, attention_mask=attention_mask)
        h = out.last_hidden_state
        if self.pooling == "mean":
            m = attention_mask.unsqueeze(-1).to(h.dtype)
            pooled = (h * m).sum(dim=1) / m.sum(dim=1).clamp(min=1e-6)
        else:

            last_idx = attention_mask.sum(dim=1) - 1
            pooled = h[torch.arange(h.size(0), device=h.device), last_idx]
        logits = self.head(self.dropout(pooled))






        logits = 5.0 * torch.sigmoid(logits)
        return ModelOutput(logits=logits)


def build_tiny_backbone(model_key, source, tokenizer):
    from transformers import DebertaV2Model, Qwen2Model
    cfg = shrink_config(AutoConfig.from_pretrained(source), tokenizer=tokenizer)
    if model_key == "qwen":
        cfg.use_cache = False
        return Qwen2Model(cfg)
    return DebertaV2Model(cfg)



_AUTO_PRETRAINED_PARAMS = set(inspect.signature(AutoModel.from_pretrained).parameters)
_DTYPE_KW = "dtype" if "dtype" in _AUTO_PRETRAINED_PARAMS else "torch_dtype"
print("from_pretrained 的精度参数名:", _DTYPE_KW)


def load_backbone():
    mcfg = MODEL_REGISTRY[MODEL_KEY]
    if SMOKE:
        return build_tiny_backbone(MODEL_KEY, MODEL_SOURCE, tokenizer)

    backbone, last_err = None, None
    for impl in mcfg["attn_impls"]:
        try:
            backbone = AutoModel.from_pretrained(MODEL_SOURCE, **{_DTYPE_KW: torch.float32},
                                                 attn_implementation=impl)
            print(f"主干加载成功: {mcfg['model_id']} | attn_implementation={impl}")
            break
        except ValueError as e:
            if "attention implementation" not in str(e).lower():
                raise
            last_err = e
            print(f"attn_implementation={impl} 不支持该架构，自动回退")
    if backbone is None:
        raise last_err

    backbone.config.use_cache = False
    return backbone






In [ ]:

FIT_SCOPE_FRAC = 1.0


def _find_layer_list(backbone):
    for path in ("encoder.layer", "layers", "layer", "decoder.layers", "encoder.decoder.layer"):
        obj = backbone
        for part in path.split("."):
            if not hasattr(obj, part):
                obj = None
                break
            obj = getattr(obj, part)
        if obj is not None and isinstance(obj, (nn.ModuleList, list)) and len(obj) > 0:
            return obj
    return None


def apply_fit_scope(model, scope_frac=None):
    frac = FIT_SCOPE_FRAC if scope_frac is None else scope_frac
    if frac >= 1.0:
        return model
    layers = _find_layer_list(model.backbone)
    if layers is None:
        print(f"⚠️ 找不到主干的层列表，本档位（只训顶部 {frac:.0%} 层）无法生效 → 退回全参微调")
        return model
    n_total = len(layers)
    n_keep = max(1, int(round(n_total * frac)))
    for p in model.backbone.parameters():
        p.requires_grad_(False)
    for layer in layers[n_total - n_keep:]:
        for p in layer.parameters():
            p.requires_grad_(True)
    n_tr = sum(p.numel() for p in model.parameters() if p.requires_grad)
    n_all = sum(p.numel() for p in model.parameters())
    print(f"档位生效：只训练顶部 {n_keep}/{n_total} 层 | 可训练参数 {n_tr / 1e6:.1f}M / "
          f"{n_all / 1e6:.1f}M ({n_tr / n_all:.1%})")
    return model

In [ ]:

def build_model():
    bb = load_backbone()
    m = CompositionalityRegressor(
        backbone=bb,
        hidden_size=bb.config.hidden_size,
        n_outputs=N_OUTPUTS,
        pooling=MODEL_REGISTRY[MODEL_KEY]["pooling"],
    ).to(DEVICE)
    apply_fit_scope(m)
    return m


model = build_model()
n_all = sum(p.numel() for p in model.parameters())
n_head = sum(p.numel() for p in model.head.parameters())
print(f"主干       : {MODEL_REGISTRY[MODEL_KEY]['model_id']}（SMOKE={SMOKE}）")
print(f"池化方式   : {MODEL_REGISTRY[MODEL_KEY]['pooling']}")
print(f"输出维度   : {N_OUTPUTS}（统一 2 列）")
print(f"总参数     : {n_all / 1e6:.1f}M | 回归头 {n_head / 1e3:.1f}K ({n_head / n_all:.4%})")
print(f"设备       : {DEVICE}")


model.eval()
with torch.no_grad():
    _b = collator([_demo_run["train_ds"][i] for i in range(2)])
    _out = model(input_ids=_b["input_ids"].to(DEVICE),
                 attention_mask=_b["attention_mask"].to(DEVICE))
print("前向 logits shape:", tuple(_out.logits.shape))
print("是否含 NaN:", bool(torch.isnan(_out.logits).any().item()))

## Step5 创建评估函数

**这一节要解决什么**：把官方的三项指标**逐字实现**出来，并让本地实现和
未来用 Trial 的 `scores.txt` 对得上。

| 指标 | 官方口径 | 一句话含义 |
|---|---|---|
| 指标 1 | 组合性相关 ρ | 全部例句上 Spearman(预测, 真值)——整体排序对不对 |
| 指标 2 | 语境方差相关 ρ | 每个目标跨例句算 std，再在**目标之间** Spearman(预测 std, 人工 std)——模型有没有真的「因句而异」 |
| 指标 3 | RMSE | 全部例句上的均方根误差——数值校准准不准 |

**指标 2 为什么是硬门槛**：如果模型对同一目标的所有例句给出同一个分数，
那它的组内 std 恒为 0，相关系数的输入数组是常数 → **相关系数未定义（nan）**。
所以指标 2 不是「加分项」，而是**及格线**：一个常数输出系统直接出局。

**指标 1 与指标 2 存在张力**：指标 1 奖励跨目标的数值校准，
指标 2 奖励目标内部的离散程度。过度压缩输出方差 → 指标 2 掉分；
过度放大方差 / 加噪声 → 指标 1 的排序被破坏、RMSE 变差。
本 notebook 不做后处理调参（那属于刷分），而是让模型真正对语境敏感。

**这一节末尾有一组断言自检**：用「完美预测 / 常数预测 / 反向预测」三种人造系统
把指标函数验一遍。指标实现一旦写错，训练完才发现就太晚了。

In [ ]:


_demo_v = _demo_run["val_df"]
_demo_key = SUBSET_SPECS["en-nn"]["key"]
_type_level_pred = _demo_v.groupby(_demo_key)["ModAvg"].transform("mean")
_gold = _demo_v["ModAvg"].to_numpy()
_g = _demo_v[_demo_key].to_numpy()

_df = pd.DataFrame({"g": _g, "p": _type_level_pred.to_numpy(), "y": _gold})
_ps = _df.groupby("g")["p"].std(ddof=1)
_gs = _df.groupby("g")["y"].std(ddof=1)
_ok = _ps.notna() & _gs.notna()
print(f"type-level 基线：指标1 Spearman = {stats.spearmanr(_type_level_pred, _gold)[0]:.3f}"
      f" | 指标3 RMSE = {np.sqrt(((_type_level_pred.to_numpy() - _gold) ** 2).mean()):.3f}")
print(f"组内预测 std 的唯一取值个数 = {_ps[_ok].nunique()}（0 表示每个目标内完全相同）")
print("→ 指标 2 的输入是常数数组，相关系数**未定义（nan）**：该系统在指标 2 上直接出局。")
print("→ 这就是为什么本任务必须「逐例句预测」，不能先聚合成目标级分数再回归。")

In [ ]:

def strip_eval_prefix(metrics):
    return {k[len("eval_"):] if k.startswith("eval_") else k: v for k, v in metrics.items()}


def _three_metrics(p, g, grp):


    with warnings.catch_warnings():
        warnings.simplefilter("ignore", category=ConstantInputWarning)
        m1 = float(stats.spearmanr(p, g)[0])
        frame = pd.DataFrame({"g": grp, "p": p, "y": g})
        ps = frame.groupby("g")["p"].std(ddof=1)
        gs = frame.groupby("g")["y"].std(ddof=1)
        ok = ps.notna() & gs.notna()
        if ok.sum() > 2 and ps[ok].nunique() > 1:
            m2 = float(stats.spearmanr(ps[ok], gs[ok])[0])
        else:
            m2 = float("nan")
    rmse = float(np.sqrt(((p - g) ** 2).mean()))
    return m1, m2, rmse, int((ps[ok] == 0).sum())


def compute_compartment_metrics(pred, gold, group_index, subset_ids, eval_cols):
    pred = np.asarray(pred, dtype=float).reshape(len(pred), -1)
    gold = np.asarray(gold, dtype=float).reshape(len(gold), -1)
    group = np.asarray(group_index)
    subset_ids = np.asarray(subset_ids)
    out = {}
    _m1s, _m2s, _rmses, _ndegs = [], [], [], []
    for name, cname, j in eval_cols:
        m = subset_ids == name
        if m.sum() < 3:
            continue
        m1, m2, rmse, ndeg = _three_metrics(pred[m, j], gold[m, j], group[m])
        out[f"{name}|{cname}_指标1_ρ"] = m1
        out[f"{name}|{cname}_指标2_ρ"] = m2
        out[f"{name}|{cname}_指标3_RMSE"] = rmse
        out[f"{name}|{cname}_组内std退化目标数"] = ndeg
        out[f"{name}|{cname}_样例数"] = int(m.sum())
        _m1s.append(m1); _m2s.append(m2); _rmses.append(rmse); _ndegs.append(ndeg)

    def _nanmean(vals):
        v = np.asarray([x for x in vals if np.isfinite(x)], dtype=float)
        return float(v.mean()) if len(v) else float("nan")

    out["合计|_指标1_ρ"] = _nanmean(_m1s)
    out["合计|_指标2_ρ"] = _nanmean(_m2s)
    out["合计|_指标3_RMSE"] = _nanmean(_rmses)
    out["合计|_组内std退化目标数"] = int(np.sum(_ndegs)) if _ndegs else 0
    out["合计|_评测设置数"] = len(_m1s)
    out["合计|_样例数"] = int(len(pred))
    out["合计|_目标数"] = int(len(np.unique(group)))
    return out


def make_compute_metrics(eval_group_index, eval_subset_index, eval_cols, subset_names):
    groups = np.asarray(eval_group_index)






    _max_id = int(np.max(eval_subset_index)) if len(eval_subset_index) else -1
    assert _max_id < len(subset_names), (
        f"subset_index 最大值 {_max_id} 超出 subset_names 长度 {len(subset_names)}："
        "subset_index 是全局编号，subset_names 必须传全局的 SUBSET_ORDER")
    subsets = np.asarray([subset_names[i] for i in eval_subset_index])

    def compute_metrics(eval_preds):
        preds, labels = eval_preds
        preds = np.asarray(preds)
        labels = np.asarray(labels)
        if preds.ndim == 1:
            preds = preds.reshape(-1, 1)
        if labels.ndim == 1:
            labels = labels.reshape(-1, 1)
        return compute_compartment_metrics(preds, labels, groups, subsets, eval_cols)

    return compute_metrics

In [ ]:




_gold = ds_labels(_demo_run["val_ds"])
_grp = np.asarray(_demo_run["val_ds"]["group_index"])
_sub = np.asarray(["en-nn"] * len(_grp))
_cols = _demo_run["eval_cols"]


_m = compute_compartment_metrics(_gold, _gold, _grp, _sub, _cols)
for _c in _cols:
    _k1 = f"{_c[0]}|{_c[1]}_指标1_ρ"
    _k2 = f"{_c[0]}|{_c[1]}_指标2_ρ"
    _k3 = f"{_c[0]}|{_c[1]}_指标3_RMSE"
    assert abs(_m[_k1] - 1.0) < 1e-9, f"完美预测的指标1 应为 1，实际 {_m[_k1]}"
    assert abs(_m[_k2] - 1.0) < 1e-9, f"完美预测的指标2 应为 1，实际 {_m[_k2]}"
    assert _m[_k3] < 1e-9, f"完美预测的 RMSE 应为 0，实际 {_m[_k3]}"
print("① 完美预测      : 指标1=1.000 指标2=1.000 RMSE=0.000  ✓")


_m = compute_compartment_metrics(np.full_like(_gold, 3.0), _gold, _grp, _sub, _cols)
for _c in _cols:
    _k1 = f"{_c[0]}|{_c[1]}_指标1_ρ"
    _k2 = f"{_c[0]}|{_c[1]}_指标2_ρ"
    assert np.isnan(_m[_k1]), f"常数预测的指标1 应为 nan，实际 {_m[_k1]}"
    assert np.isnan(_m[_k2]), f"常数预测的指标2 应为 nan，实际 {_m[_k2]}"
print("② 常数预测      : 指标1=nan 指标2=nan（未定义）              ✓")


_m = compute_compartment_metrics(-_gold, _gold, _grp, _sub, _cols)
for _c in _cols:
    _k1 = f"{_c[0]}|{_c[1]}_指标1_ρ"
    assert abs(_m[_k1] + 1.0) < 1e-9, f"反向预测的指标1 应为 -1，实际 {_m[_k1]}"
print("③ 反向预测      : 指标1=-1.000                              ✓")


_flat = np.full_like(_gold, 2.5)
_m = compute_compartment_metrics(_gold, _flat, _grp, _sub, _cols)
for _c in _cols:
    _k2 = f"{_c[0]}|{_c[1]}_指标2_ρ"
    assert np.isnan(_m[_k2]), f"人工 std 恒为 0 时指标2 应为 nan，实际 {_m[_k2]}"
print("④ 人工 std 恒为 0: 指标2=nan（被解释变量无方差）           ✓")
print("→ 指标实现通过四组断言，接下来所有结果都由它产出。")

## Step6 配置训练参数

**这一节要解决什么**：把超参组装成一个**函数**，而不是散落的字面量。

**为什么必须是函数**：transformers 的 `Trainer` 会**就地改写** `TrainingArguments` 对象
（`per_device_train_batch_size`、`max_steps`、`label_names`、`_n_gpu` 都会被动过）。
如果两个 Trainer 共用同一个 args 对象，第二个 Trainer 拿到的就是「已经被训练过」的脏 args，
它的 `num_batches` 与实际 dataloader 长度对不上，直接抛
`ValueError: zip() argument 2 is longer than argument 1`。
所以「超参相同」的正确实现是：**每次调用 `build_training_args()` 重新构造一个对象**，
字段默认值相同即等价，而不是复用同一个变量。

**超参取值与理由**：

| 超参 | 值 | 为什么 |
|---|---|---|
| `learning_rate` | 2e-5 | 微调大模型的常规档位。两个模型**用同一个值**，对照实验里 lr 不成为混淆变量 |
| `per_device_train_batch_size` | 8 | 序列长 160、模型 900M～1.5B；显存不够时用 `gradient_accumulation_steps` 把有效 batch 补回去（数学等价、几乎无损，第一选择） |
| `num_train_epochs` | 3 | 本项目每个目标只有 5–10 条例句，全量数据仅 1.5k–3.5k 条；轮数太多会过拟合到「记住句子」 |
| `warmup_steps` | 总步数 × 10% | transformers 5 **删除了 `warmup_ratio`**，只保留绝对步数，所以这里自己算 |
| `lr_scheduler_type` | linear | 线性衰减是回归任务的稳定默认；余弦/常数留到消融时再比 |
| `weight_decay` | 0.01 | 常规正则量级 |
| `eval_strategy` / `save_strategy` | epoch | transformers 5 把 `evaluation_strategy` 改名为 `eval_strategy` |
| `load_best_model_at_end` | True | 必须与 `save_strategy="epoch"` 同时满足，否则初始化阶段直接抛 ValueError |

In [ ]:






_TA_PARAMS = set(inspect.signature(TrainingArguments.__init__).parameters)
_EVAL_STRATEGY_KEY = "eval_strategy" if "eval_strategy" in _TA_PARAMS else "evaluation_strategy"
_TRAINER_PARAMS = set(inspect.signature(Trainer.__init__).parameters)
_TOKENIZER_KW = "processing_class" if "processing_class" in _TRAINER_PARAMS else "tokenizer"
_HAS_SAVE_ONLY_MODEL = "save_only_model" in _TA_PARAMS
print("transformers 兼容层:", {"eval_key": _EVAL_STRATEGY_KEY, "tokenizer_kw": _TOKENIZER_KW,
                                "save_only_model": _HAS_SAVE_ONLY_MODEL})



def _bnb_available():
    try:
        import bitsandbytes
        return True
    except Exception:
        return False


_HAS_BNB = _bnb_available()


def _use_8bit_allowed():
    mode = USE_8BIT_OPT
    if isinstance(mode, str):
        low = mode.strip().lower()
        if low in ("true", "1", "yes", "8bit"):
            return True
        if low in ("false", "0", "no", "fp32"):
            return False
        return _HAS_BNB
    return bool(mode)






























FIT_ACT_MIN_GB = 0.3

FIT_TIERS = [
    {"name": "t0", "label": "全参微调 · fp32 AdamW · AMP", "optim": "adamw_torch",
     "amp": True, "scope": 1.00, "ckpt": False, "act_gb": 2.0},
    {"name": "t1", "label": "全参微调 · 8bit AdamW · AMP", "optim": "adamw_bnb_8bit",
     "amp": True, "scope": 1.00, "ckpt": False, "act_gb": 2.0},
    {"name": "t2", "label": "全参微调 · Adafactor · AMP", "optim": "adafactor",
     "amp": True, "scope": 1.00, "ckpt": True, "act_gb": 1.5},
    {"name": "t3", "label": "全参微调 · Adafactor · 无 AMP", "optim": "adafactor",
     "amp": False, "scope": 1.00, "ckpt": True, "act_gb": 1.5},
    {"name": "t4", "label": "顶部 50% 层 · Adafactor · AMP", "optim": "adafactor",
     "amp": True, "scope": 0.50, "ckpt": True, "act_gb": 1.5},
    {"name": "t5", "label": "顶部 50% 层 · Adafactor · 无 AMP", "optim": "adafactor",
     "amp": False, "scope": 0.50, "ckpt": True, "act_gb": 1.5},
    {"name": "t6", "label": "顶部 25% 层 · Adafactor · 无 AMP", "optim": "adafactor",
     "amp": False, "scope": 0.25, "ckpt": True, "act_gb": 1.5},
]

_OPTIM_STATE_BYTES = {"adamw_torch": 8.0, "adamw_bnb_8bit": 2.0, "adafactor": 0.2}


def tier_fixed_bytes(tier):
    w = 4.0
    amp = 2.0 if tier["amp"] else 0.0
    st = _OPTIM_STATE_BYTES[tier["optim"]]
    return w + amp + tier["scope"] * (4.0 + st)


def tier_estimate_gb(n_params, tier, act_gb=None):
    fixed = n_params * tier_fixed_bytes(tier) / (1024 ** 3)
    act = tier["act_gb"] if act_gb is None else act_gb
    return fixed, fixed + act


def _tier_available(tier):
    if tier["optim"] == "adamw_bnb_8bit":
        return _use_8bit_allowed() and _HAS_BNB
    return True


def _vram_total_gb():
    if not torch.cuda.is_available():
        return None
    return torch.cuda.get_device_properties(torch.cuda.current_device()).total_memory / (1024 ** 3)


def resolve_fit_tier(vram_gb, ref_params):
    if vram_gb is None:
        return FIT_TIERS[0], []
    budget = vram_gb * FIT_SAFETY
    rows, chosen = [], None
    for t in FIT_TIERS:
        if not _tier_available(t):
            rows.append((t["name"], t["label"], None, None, None,
                         "不可用（需要 bitsandbytes）" if t["optim"] == "adamw_bnb_8bit" else "不可用"))
            continue

        fixed, need_min = tier_estimate_gb(ref_params, t, act_gb=FIT_ACT_MIN_GB)
        _fixed8, need8 = tier_estimate_gb(ref_params, t)
        ok = need_min <= budget
        rows.append((t["name"], t["label"], fixed, need_min, need8,
                     "✓ 装得下" if ok else f"✗ 超出预算 {budget:.1f} GiB"))
        if ok and chosen is None:
            chosen = t

    if FIT_MODE != "auto":
        forced = next((t for t in FIT_TIERS if t["name"] == str(FIT_MODE).lower()), None)
        if forced is not None and _tier_available(forced):
            chosen = forced
    if chosen is None:

        avail = [t for t in FIT_TIERS if _tier_available(t)]
        chosen = avail[-1] if avail else FIT_TIERS[0]
    return chosen, rows


FIT_TIER_NAME = FIT_TIERS[0]["name"]
OPTIM = FIT_TIERS[0]["optim"]
USE_GRAD_CKPT = False
FIT_SCOPE_FRAC = 1.0


def apply_tier(tier):
    global OPTIM, USE_GRAD_CKPT, FIT_SCOPE_FRAC, FIT_TIER_NAME, USE_BF16, USE_FP16
    OPTIM = tier["optim"]
    USE_GRAD_CKPT = tier["ckpt"]
    FIT_SCOPE_FRAC = tier["scope"]
    FIT_TIER_NAME = tier["name"]
    if tier["amp"]:
        USE_BF16, USE_FP16 = HW_BF16, HW_FP16
    else:
        USE_BF16, USE_FP16 = False, False


def check_vram_budget(n_params, model_name, optim=None):
    if not torch.cuda.is_available():
        print("非 CUDA 环境：跳过显存预算检查（按标准全参微调跑）")
        return None
    n_gpu = torch.cuda.device_count()
    for _i in range(n_gpu):
        _p = torch.cuda.get_device_properties(_i)
        print(f"  GPU{_i}: {_p.name} | {_p.total_memory / 1024 ** 3:.1f} GiB | "
              f"compute capability {_p.major}.{_p.minor}")
    total_gb = _vram_total_gb()
    print(f"显存预算：{model_name} | {n_params / 1e6:.0f}M 参数（实际） | "
          f"选档参考 {FIT_REF_PARAMS / 1e6:.0f}M（两条基线取较大者，保证同档） | "
          f"单卡容量 {total_gb:.1f} GiB | 安全系数 {FIT_SAFETY:.0%}")
    tier, rows = resolve_fit_tier(total_gb, FIT_REF_PARAMS)
    print("  档位表（固定开销与 batch 无关，降 batch 救不了）：")
    print(f"    {'档':4s}{'含义':34s}{'固定':>9s}{'bs=1 需':>10s}{'bs=8 需':>10s}  判定")
    for name, label, fixed, need_min, need8, verdict in rows:
        if fixed is None:
            print(f"    {name:4s}{label:34s}{'-':>9s}{'-':>10s}{'-':>10s}  {verdict}")
        else:
            print(f"    {name:4s}{label:34s}{fixed:8.1f}G{need_min:9.1f}G{need8:9.1f}G  {verdict}")
    fixed_act, need_act = tier_estimate_gb(n_params, tier)
    print(f"  → 采用档位 {tier['name']}（{tier['label']}）；"
          f"本模型固定开销 {fixed_act:.1f} GiB"
          f"（+ 激活：bs=1 约 {tier['act_gb'] * 0.2:.1f} / bs=8 约 {tier['act_gb']:.1f} GiB，"
          f"实际能跑多大由探测决定）")
    if n_gpu > 1:
        print(f"  ⚠️ 本机有 {n_gpu} 张卡，但**单进程默认只用 GPU0**。多卡要用对方式才有效：")
        print("     · DDP（数据并行）：每张卡各存**一整份**模型 → **不降低单卡显存需求**，只是把数据分半")
        print("     · ZeRO-3 / FSDP（参数分片）：把参数+梯度+优化器状态切到多卡 → "
              "这才能真正用满多卡显存（需装 deepspeed）")
        print("     · 最省事：两条基线**各占一张卡并行跑**（两个独立实验，互不干扰）")
    _fixed_m, need_min = tier_estimate_gb(n_params, tier, act_gb=FIT_ACT_MIN_GB)
    if need_min <= total_gb:
        return True
    print("=" * 78)
    print(f"⚠️ 警告：档位 {tier['name']} 即使按 micro-batch=1 也要约 {need_min:.1f} GiB，"
          f"本卡只有 {total_gb:.1f} GiB。")
    print("   Step8 会用真实模型逐档试探；若仍 OOM 会自动再降一档，"
          "并在全部档位都失败时给出精确诊断（而不是一句 out of memory）。")
    print("=" * 78)
    return False


def build_training_args(output_dir, n_train, **overrides):
    steps_per_epoch = math.ceil(n_train / (BATCH_SIZE * GRAD_ACCUM))
    total_steps = steps_per_epoch * EPOCHS
    base = dict(
        output_dir=str(output_dir),
        per_device_train_batch_size=BATCH_SIZE,
        per_device_eval_batch_size=BATCH_SIZE * 2,
        gradient_accumulation_steps=GRAD_ACCUM,
        learning_rate=LR,
        num_train_epochs=EPOCHS,
        weight_decay=WEIGHT_DECAY,
        warmup_steps=max(1, int(total_steps * WARMUP_RATIO)),
        lr_scheduler_type="linear",
        logging_steps=max(1, steps_per_epoch // 2),
        save_strategy="epoch",
        save_total_limit=1,




        load_best_model_at_end=True,
        metric_for_best_model="合计|_指标1_ρ",
        greater_is_better=True,
        remove_unused_columns=False,
        dataloader_pin_memory=(DEVICE == "cuda"),
        dataloader_num_workers=NUM_WORKERS,
        dataloader_persistent_workers=DATALOADER_PERSISTENT,
        dataloader_multiprocessing_context=None if NUM_WORKERS == 0 else "fork",
        dataloader_prefetch_factor=None if NUM_WORKERS == 0 else 4,
        seed=SEED,
        data_seed=SEED,
        report_to=[],
        disable_tqdm=True,
        bf16=USE_BF16,
        fp16=USE_FP16,
        optim=OPTIM,
        gradient_checkpointing=USE_GRAD_CKPT,
    )
    base[_EVAL_STRATEGY_KEY] = "epoch"
    if _HAS_SAVE_ONLY_MODEL:
        base["save_only_model"] = True
    base.update(overrides)






    if base.get("gradient_checkpointing") and "gradient_checkpointing_kwargs" in _TA_PARAMS:
        base["gradient_checkpointing_kwargs"] = {"use_reentrant": False}
    args = TrainingArguments(**base)












    try:
        args._n_gpu = 1
    except Exception:
        pass
    return args

## Step7 创建训练器

**这一节要解决什么**：写出损失函数，并把它接进 `Trainer`。

**损失为什么是三段式**（本项目实测得出的结论）：

```
L = L_huber(绝对值)  +  λ · L_rank(组内成对排序)  +  μ · L_std(组内 std 匹配)
```

| 项 | 作用 | 为什么需要 |
|---|---|---|
| `L_huber` | 回归到真实分数 | 提供绝对值的锚。Huber 在小误差处是 L2（梯度平滑）、大误差处是 L1（对离群稳健），比纯 MSE 更适合有界的 0–5 标签 |
| `L_rank` | 组内相对顺序 | 本项目实测标签的**均值**噪声方差是 `Std²/10`（因为每个标签是 10 个整数评分的均值），绝对值回归会被噪声带偏；成对排序只关心组内相对顺序，**对噪声不敏感**，且天然服务于指标 2 |
| `L_std` | 组内离散程度 | 指标 2 的被解释变量就是「每个目标跨例句的 std」。显式让预测的组内 std 靠近人工 std，是直接对着指标 2 优化 |

**为什么默认 `λ = μ = 0`**：本 notebook 交付的是**基线**，
基线的定义是「最简单、没有额外技巧的那一版」。三个开关都留好，
后续做消融时只改两个数字，其余管线一行不动。

**为什么只覆写 `compute_loss` 这一个方法**：`Trainer` 把「损失怎么算」和
「怎么优化、怎么调度、怎么保存、怎么评估」分得很干净。
前向、反向、梯度累积、日志、评估、保存全由框架负责，我们只替换损失这一环。

In [ ]:



toyl = torch.tensor([[1.0], [4.0], [2.0], [5.0]])
toyp = torch.tensor([[1.5], [3.0], [2.2], [4.0]])
toyg = torch.tensor([[0], [0], [1], [1]])

_l_huber = F.huber_loss(toyp, toyl, delta=HUBER_DELTA)

_l_rank = torch.tensor(0.0)
for a, b in [(0, 1), (2, 3)]:
    _s = torch.sign(toyl[a] - toyl[b])
    _l_rank = _l_rank + F.softplus(-_s * (toyp[a] - toyp[b]))
_l_rank = _l_rank / 2

_l_std = torch.tensor(0.0)
for g in torch.unique(toyg):
    _m = toyg == g
    if _m.sum() >= 2:
        _l_std = _l_std + (toyp[_m].std(unbiased=True) - toyl[_m].std(unbiased=True)) ** 2
print(f"L_huber = {_l_huber.item():.4f}   绝对值误差")
print(f"L_rank  = {_l_rank.item():.4f}   组内相对顺序（目标0真实 1<4，预测 1.5>3.0 → 被罚）")
print(f"L_std   = {_l_std.item():.4f}   组内离散程度（人工组内 std=3.0，预测组内 std=1.5 → 被罚）")
print("→ 三项各自惩罚不同的东西；λ、μ 为 0 时只有第一项生效，即纯回归基线。")

In [ ]:

def pairwise_rank_loss(pred, gold, group_index, tau=1.0):
    B = pred.shape[0]
    same = group_index[:, None] == group_index[None, :]
    iu = torch.triu(torch.ones(B, B, dtype=torch.bool, device=pred.device), diagonal=1)
    mask = same & iu
    if mask.sum() == 0:
        return pred.sum() * 0.0
    diff_p = pred[:, None] - pred[None, :]
    diff_y = gold[:, None] - gold[None, :]
    sign = torch.sign(diff_y)
    mask = mask & (sign != 0)
    if mask.sum() == 0:
        return pred.sum() * 0.0
    return F.softplus(-sign[mask] * diff_p[mask] / tau).mean()


def group_std_loss(pred, gold, group_index):
    losses = []
    for g in torch.unique(group_index):
        m = group_index == g
        if int(m.sum().item()) >= 2:
            losses.append((pred[m].std(unbiased=True) - gold[m].std(unbiased=True)) ** 2)
    if not losses:
        return pred.sum() * 0.0
    return torch.stack(losses).mean()


class CompartmentTrainer(Trainer):

    def __init__(self, *args, grouped_batching=True, **kwargs):
        super().__init__(*args, **kwargs)
        self.grouped_batching = grouped_batching

    def _get_train_sampler(self, train_dataset=None):
        if not self.grouped_batching:
            return super()._get_train_sampler(train_dataset)
        ds = train_dataset if train_dataset is not None else self.train_dataset
        return TargetGroupedSampler(ds["group_index"], seed=self.args.seed)

    def compute_loss(self, model, inputs, return_outputs=False, num_items_in_batch=None):
        labels = inputs["labels"]
        mask = inputs.get("loss_mask")
        gidx = inputs.get("group_index")
        outputs = model(input_ids=inputs["input_ids"],
                        attention_mask=inputs["attention_mask"])
        logits = outputs.logits



        per = F.huber_loss(logits, labels, delta=HUBER_DELTA, reduction="none")
        if mask is not None:
            loss = (per * mask).sum() / mask.sum().clamp(min=1.0)
        else:
            loss = per.mean()
        if gidx is not None and (LAMBDA_RANK > 0 or MU_STD > 0):
            for j in range(logits.shape[1]):
                if LAMBDA_RANK > 0:
                    loss = loss + LAMBDA_RANK * pairwise_rank_loss(logits[:, j], labels[:, j], gidx)
                if MU_STD > 0:
                    loss = loss + MU_STD * group_std_loss(logits[:, j], labels[:, j], gidx)
        return (loss, {"logits": logits}) if return_outputs else loss


class TargetGroupedSampler(Sampler):
    def __init__(self, group_index, seed=SEED, shuffle=True):
        self.groups = np.asarray(group_index)
        self.seed = seed
        self.shuffle = shuffle
        self.epoch = 0
        self._indices = self._build()

    def _build(self):
        rng = np.random.default_rng(self.seed + self.epoch)
        uniq = np.unique(self.groups)
        if self.shuffle:
            rng.shuffle(uniq)
        order = []
        for g in uniq:
            idx = np.where(self.groups == g)[0]
            if self.shuffle:
                rng.shuffle(idx)
            order.extend(idx.tolist())
        return order

    def __iter__(self):
        return iter(self._indices)

    def __len__(self):
        return len(self._indices)

    def set_epoch(self, epoch):
        self.epoch = epoch
        self._indices = self._build()


print("CompartmentTrainer / TargetGroupedSampler 已定义")

## Step8 模型训练（自动跑完全部运行）

**这一节不需要任何手动切换**：它按 Step1 的 `RUN_PLAN` 自动依次执行
「四个子集各单独训练一次」+「四个子集联合训练一次」，共 5 次。

**每一次运行都做四件事**（顺序不能换）：

1. **准备数据**：按本次参与的子集切分、构造 Dataset（`prepare_run`）。
2. **训练前评估随机初始化模型** —— 给 Step9 的「参考对照」区提供一条基线。
   少了它，「训练后 ρ=0.6」无法回答「它到底学到了什么」。
3. **正式训练**。
4. **训练后评估**：与随机初始化**完全相同的口径**（同一份验证集、同一个
   `compute_metrics`），唯一区别是模型权重。口径不对齐的对照没有意义。

**第一轮额外做三项开训前检查**（只做一次，因为它们与数据规模无关）：

- **显存探测**：逐档真跑 2 步，量峰值显存，取能跑通的最大 micro-batch，
  再用梯度累积把有效 batch 补回 8。
- **梯度流验证**：手动 `backward()` 一次，打印「有多少参数真的拿到了梯度」。
  能抓到的典型 bug 是「主干被冻结 / 回归头没接上」——这些**不会报错**，
  只会让模型训半天纹丝不动。
- **冒烟测试**：`max_steps=5` 跑通「前向 → 损失 → 反向 → 参数更新 → 评估」
  整条链路。直接开长训，一旦中途报错，要么白等几十分钟，要么分不清是 bug 还是超时。

**为什么每次运行都要重建模型**：冒烟测试那 5 步、显存探测那 2 步探针都会改动权重。
正式训练必须从**干净的随机初始化**开始，否则基线不干净、不可复现。

In [ ]:

def make_trainer(m, out_dir, data, **overrides):
    _args = build_training_args(out_dir, len(data["train_ds"]), **overrides)






    _n_gpu = int(getattr(_args, "n_gpu", 1) or 1)
    assert _n_gpu == 1, (
        f"args.n_gpu={_n_gpu} > 1：Trainer 会自动套 nn.DataParallel，"
        "其反向传播会复制一整份梯度导致 OOM。请用 COMPARTMENT_GPU=<卡号> 只暴露一张卡。")
    return CompartmentTrainer(
        model=m,
        args=_args,
        train_dataset=data["train_ds"],
        eval_dataset=data["val_ds"],
        data_collator=collator,


        **{_TOKENIZER_KW: tokenizer},
        compute_metrics=make_compute_metrics(
            data["val_ds"]["group_index"], data["val_ds"]["subset_index"],
            data["eval_cols"], SUBSET_ORDER),







        grouped_batching=GROUPED_BATCHING,
    )
















class FitOOMError(RuntimeError):
    pass


def _is_oom(exc):
    return "out of memory" in str(exc).lower()


def probe_fit_tier(model, data, tier, candidates=(8, 4, 2, 1), steps=2):
    if not torch.cuda.is_available():
        print("非 CUDA 环境（CPU/MPS）：跳过显存探测，沿用配置中的 batch size")
        return None
    for bs in candidates:
        _tr = None
        try:
            _tr = make_trainer(model, OUTPUT_ROOT / "_probe", data,
                               per_device_train_batch_size=bs,
                               gradient_accumulation_steps=1,
                               max_steps=steps,
                               save_strategy="no",
                               load_best_model_at_end=False,
                               logging_steps=steps,
                               optim=tier["optim"],
                               gradient_checkpointing=tier["ckpt"],
                               dataloader_num_workers=0, dataloader_persistent_workers=False,
                               dataloader_multiprocessing_context=None,
                               dataloader_prefetch_factor=None,
                               **{_EVAL_STRATEGY_KEY: "no"})
            torch.cuda.reset_peak_memory_stats()
            _t0 = time.time()
            _tr.train()
            _dt = time.time() - _t0
            peak = torch.cuda.max_memory_allocated() / (1024 ** 3)


            sps = _dt / max(1, steps * bs)
            print(f"  {tier['name']} micro-batch={bs}: 通过，峰值显存 {peak:.2f} GiB，"
                  f"每样本 {sps * 1000:.2f} ms")
            return bs, peak, sps
        except RuntimeError as e:
            if not _is_oom(e):
                raise


            _lines = [l for l in str(e).strip().splitlines() if l.strip()]
            _head = _lines[0][:150] if _lines else ""
            _detail = next((l for l in _lines if "Tried to allocate" in l or "GiB is free" in l), "")
            print(f"  {tier['name']} micro-batch={bs}: OOM | {_head}")
            if _detail:
                print(f"      {_detail.strip()[:220]}")
        finally:
            if _tr is not None:
                del _tr
            gc.collect()
            torch.cuda.empty_cache()
    return None



def _total_train_rows():
    tot = 0
    for _name in SUBSET_ORDER:
        _p = DATA_ROOT / f"{_name}-train.tsv"
        with open(_p, "r", encoding="utf-8") as _f:
            tot += max(0, sum(1 for _ in _f) - 1)
    return tot


def project_train_time(sec_per_sample):
    n_rows = _total_train_rows()
    per_epoch = 0.8 * n_rows * 2
    total = per_epoch * EPOCHS
    hours = total * sec_per_sample * 1.3 / 3600
    return n_rows, total, hours


def prepare_for_training(build_fn, data):
    vram = _vram_total_gb()
    tier0, _rows = resolve_fit_tier(vram, FIT_REF_PARAMS)
    start = FIT_TIERS.index(tier0)
    last_err = ""
    for i in range(start, len(FIT_TIERS)):
        t = FIT_TIERS[i]
        if not _tier_available(t):
            print(f"跳过档位 {t['name']}：当前环境不可用")
            continue
        apply_tier(t)
        print("=" * 78)
        print(f"试探档位 {t['name']}：{t['label']} | optim={t['optim']} | "
              f"可训层比例={t['scope']:.0%} | 混合精度={'开' if t['amp'] else '关'} | "
              f"gradient_checkpointing={t['ckpt']} | "
              f"固定开销 {tier_estimate_gb(FIT_REF_PARAMS, t)[0]:.1f} GiB（按参考 {FIT_REF_PARAMS / 1e6:.0f}M 算）")
        print("=" * 78)
        model = None
        try:
            model = build_fn()
            if PROBE_MEMORY and torch.cuda.is_available():
                res = probe_fit_tier(model, data, t)
                if res is None:
                    last_err = f"档位 {t['name']}：所有 batch 档位都 OOM"
                    print(f"→ {last_err}，自动降一档重试")
                    continue
                bs, peak, sps = res
            else:
                print("非 CUDA 环境或 PROBE_MEMORY=False：跳过显存探测，沿用配置中的 batch size")
                bs, peak, sps = BATCH_SIZE, float("nan"), float("nan")
            try:
                run_preflight_checks(model, data, OUTPUT_ROOT / "_preflight")
            except FitOOMError as e:
                last_err = f"档位 {t['name']}：开训前检查阶段仍 OOM（{str(e)[:160]}）"
                print(f"→ {last_err}，自动降一档重试")
                continue
            return t, bs, peak, sps
        finally:
            if model is not None:
                del model
            gc.collect()
            if torch.cuda.is_available():
                torch.cuda.empty_cache()

    n_params = FIT_REF_PARAMS
    _lines = [f"档位 {t['name']}：{t['label']} → 固定开销 "
              f"{tier_estimate_gb(n_params, t)[0]:.1f} GiB" for t in FIT_TIERS]
    raise RuntimeError(
        "所有训练档位都跑不通。\n"
        f"  本卡容量          : {vram:.1f} GiB\n"
        f"  参考参数量        : {n_params / 1e6:.0f}M（两条基线中较大者）\n"
        f"  最后一次失败原因  : {last_err}\n"
        "  各档位固定开销    : " + " | ".join(_lines) + "\n"
        "  可选方案：\n"
        "    ① 选更大显存的实例（如 L4 24GB / A100 40GB）—— 档位会自动回到 t0/t1\n"
        "    ② 换更小的同族模型（deberta-v3-base / Qwen2.5-0.5B-Instruct），管线一行不用改\n"
        "    ③ ZeRO-3 / FSDP 分片（deepspeed）\n"
        "    ④ 手动指定档位：把 FIT_MODE 写成 't0'~'t3'")


print("Trainer 工厂与档位探测就绪")

In [ ]:

def run_preflight_checks(model, data, out_dir):
    try:
        _preflight_body(model, data, out_dir)
    except RuntimeError as e:
        if _is_oom(e):
            raise FitOOMError(str(e)[:300]) from e
        raise


def _preflight_body(model, data, out_dir):

    model.train()
    _gb = collator([data["train_ds"][i] for i in range(4)])
    _gb = {k: (v.to(DEVICE) if isinstance(v, torch.Tensor) else v) for k, v in _gb.items()}
    model.zero_grad(set_to_none=True)
    _go = model(input_ids=_gb["input_ids"], attention_mask=_gb["attention_mask"])
    _gl = F.huber_loss(_go.logits, _gb["labels"], delta=HUBER_DELTA)
    _gl.backward()
    _named = [(n, p) for n, p in model.named_parameters() if p.requires_grad]
    _with_grad = [(n, p) for n, p in _named if p.grad is not None and float(p.grad.abs().sum()) > 0]
    _head_g = float(model.head.weight.grad.abs().sum()) if model.head.weight.grad is not None else 0.0
    _bb_p = next(model.backbone.parameters())
    _bb_g = float(_bb_p.grad.abs().sum()) if _bb_p.grad is not None else 0.0
    print(f"loss                   : {_gl.item():.4f}   （NaN? {bool(torch.isnan(_gl))}）")
    print(f"可训练参数             : {len(_named)}")
    print(f"拿到非零梯度的参数     : {len(_with_grad)}  ({len(_with_grad) / max(1, len(_named)):.1%})")
    print(f"回归头权重梯度绝对值和 : {_head_g:.6g}   （必须 > 0，否则头没接上）")
    print(f"主干首参数梯度绝对值和 : {_bb_g:.6g}   （必须 > 0，否则主干被冻结）")
    assert _head_g > 0, "回归头没有梯度：回归头没接进计算图"
    assert _bb_g > 0, "主干没有梯度：主干被冻结了"
    assert not torch.isnan(_gl), "loss 出现 NaN：多半是 fp16 加载或标签里有脏值"
    model.zero_grad(set_to_none=True)
    print("→ 梯度流验证通过。")


    _st = make_trainer(model, out_dir / "_smoke", data,
                       max_steps=5, num_train_epochs=1,
                       save_strategy="no", load_best_model_at_end=False,
                       logging_steps=1, **{_EVAL_STRATEGY_KEY: "no"})
    _so = _st.train()
    del _st
    gc.collect()
    assert np.isfinite(_so.training_loss), "冒烟训练 loss 非有限值"
    print("冒烟训练 loss 轨迹:", _so.training_loss)
    print("→ 冒烟测试通过（每个 Trainer 都是独立构造 args，不复用对象）。")

In [ ]:





ALL_RUNS = []

for _i, (_mode, _subset) in enumerate(RUN_PLAN):
    _subsets = SUBSET_ORDER if _mode == "joint" else [_subset]
    _run_name = f"{_mode}_{_subset}" if _subset else _mode
    _out_dir = OUTPUT_ROOT / MODEL_KEY / _run_name
    print("=" * 78)
    print(f"运行 {_i + 1}/{len(RUN_PLAN)}：{_run_name}  |  参与子集 {_subsets}")
    print("=" * 78)


    _data = prepare_run(_subsets, tokenizer)
    print(f"训练 {len(_data['train_ds'])} 条 | 验证 {len(_data['val_ds'])} 条 | "
          f"评测设置 {len(_data['eval_cols'])} 个")


    model = build_model()
    n_params = sum(p.numel() for p in model.parameters())
    check_vram_budget(n_params, MODEL_REGISTRY[MODEL_KEY]["model_id"])







    if _i == 0:
        model = None
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()


        _tier, _bs, _peak, _sps = prepare_for_training(build_model, _data)
        apply_tier(_tier)
        BATCH_SIZE = _bs



        GRAD_ACCUM = max(1, int(round(TARGET_EFFECTIVE_BATCH / _bs)))
        print(f"→ 采用档位 {_tier['name']}（{_tier['label']}）| micro-batch={BATCH_SIZE} | "
              f"梯度累积={GRAD_ACCUM} | 有效 batch={BATCH_SIZE * GRAD_ACCUM}"
              f"（目标 {TARGET_EFFECTIVE_BATCH}） | 峰值显存约 {_peak:.2f} GiB")
        print("   （档位由「两条基线中较大者的参数量」选定 → 同一台机器上两条基线必然同档，"
              "对照口径不会被显存差异污染）")

        if _sps == _sps:
            _n_rows, _tot, _hours = project_train_time(_sps)
            print(f"训练时长投影：5 次运行共约 {_tot:,} 个样本·轮 → 约 {_hours:.1f} 小时"
                  f"（Kaggle 上限 12 小时，预算 {TIME_BUDGET_H} 小时）")
            if _hours > TIME_BUDGET_H:
                print("=" * 78)
                print(f"⚠️ 投影时长 {_hours:.1f} 小时 > 预算 {TIME_BUDGET_H} 小时！")
                print("   ① 调小 EPOCHS（两条基线**必须同时**改，否则对照口径失效）")
                print("   ② 把 RUN_PLAN 改成只跑联合那一次（先拿到一个能提交的模型）")
                print("   ③ 换更快的卡 / 更小的同族模型")
                print("=" * 78)


    model = None
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    model = build_model()

    trainer = make_trainer(model, _out_dir, _data)


    _t0 = time.time()
    _rand_metrics = strip_eval_prefix(trainer.evaluate(_data["val_ds"]))
    print(f"随机初始化模型基线（{time.time() - _t0:.1f}s）: "
          f"指标1 ρ={_rand_metrics.get('合计|_指标1_ρ', float('nan')):.4f} | "
          f"指标2 ρ={_rand_metrics.get('合计|_指标2_ρ', float('nan')):.4f} | "
          f"RMSE={_rand_metrics.get('合计|_指标3_RMSE', float('nan')):.4f}")


    _t0 = time.time()
    _res = trainer.train()
    _dt = time.time() - _t0
    print(f"训练完成，用时 {_dt / 60:.1f} 分钟，loss={_res.training_loss:.4f}")


    _pred_out = trainer.predict(_data["val_ds"])
    _trained_metrics = strip_eval_prefix(trainer.evaluate(_data["val_ds"]))
    print("--- 训练后指标 ---")
    for k, v in _trained_metrics.items():
        if isinstance(v, float) and ("指标" in k or "目标数" in k):
            print(f"   {k:34s} {v:.4f}")


    _val_df = _data["val_df"]
    _gold = ds_labels(_data["val_ds"])
    _type_level = np.zeros((len(_val_df), N_OUTPUTS))
    for _name, _cname, _j in _data["eval_cols"]:
        _src = "y0" if _j == 0 else "y1"
        _m = (_val_df["subset_name"] == _name).to_numpy()
        _type_level[_m, _j] = _val_df.groupby("group_key")[_src].transform("mean").to_numpy()[_m]
    _oracle_metrics = compute_compartment_metrics(
        _type_level, _gold, np.asarray(_data["val_ds"]["group_index"]),



        np.asarray([SUBSET_ORDER[i] for i in _data["val_ds"]["subset_index"]]),
        _data["eval_cols"])





    if _run_name in SAVE_FINAL_FOR:
        trainer.save_model(str(_out_dir / "final"))
        tokenizer.save_pretrained(str(_out_dir / "final"))
        print(f"模型已保存到: {_out_dir / 'final'}")



    for _ck in sorted(_out_dir.glob("checkpoint-*")):
        shutil.rmtree(_ck, ignore_errors=True)





    _settings = dict(
        model_key=MODEL_KEY, model_id=MODEL_REGISTRY[MODEL_KEY]["model_id"],
        optim=OPTIM, gradient_checkpointing=USE_GRAD_CKPT,


        fit_tier=FIT_TIER_NAME, trainable_scope=FIT_SCOPE_FRAC,
        mixed_precision=("amp" if (USE_BF16 or USE_FP16) else "fp32"),
        gpu_index=GPU_INDEX,
        effective_batch=BATCH_SIZE * GRAD_ACCUM, micro_batch=BATCH_SIZE,
        grad_accum=GRAD_ACCUM, lr=LR, epochs=EPOCHS, max_length=MAX_LENGTH,
        weight_decay=WEIGHT_DECAY, warmup_ratio=WARMUP_RATIO,
        huber_delta=HUBER_DELTA, lambda_rank=LAMBDA_RANK, mu_std=MU_STD,
        seed=SEED, n_outputs=N_OUTPUTS, pooling=MODEL_REGISTRY[MODEL_KEY]["pooling"],
        train_subset=_subsets,
    )
    _set_dir = OUTPUT_ROOT / "summary"
    _set_dir.mkdir(parents=True, exist_ok=True)
    with open(_set_dir / f"run_settings_{_run_name}.json", "w", encoding="utf-8") as _f:
        json.dump(_settings, _f, ensure_ascii=False, indent=2)
    ALL_RUNS.append(dict(
        run_name=_run_name, mode=_mode, subset=_subset, subsets=list(_subsets),
        eval_cols=_data["eval_cols"], out_dir=_out_dir,
        metrics=_trained_metrics, random_metrics=_rand_metrics, oracle_metrics=_oracle_metrics,
        settings=_settings,
        preds=np.asarray(_pred_out.predictions).reshape(len(_gold), -1),
        gold=_gold,
        groups=np.asarray(_data["val_ds"]["group_index"]),
        subset_ids=np.asarray([SUBSET_ORDER[i] for i in _data["val_ds"]["subset_index"]]),
        val_df=_val_df,
        train_minutes=_dt / 60,
    ))
    print(f"→ 运行 {_run_name} 完成，产物在 {_out_dir}")


    del trainer, model, _pred_out
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

print("\n" + "=" * 78)
print(f"全部 {len(ALL_RUNS)} 次运行完成")
print("=" * 78)

## Step9 模型评估、对照图与结果分析

这一节分成**两个互不混淆的区块**：

### 区块一：评估标准（模型本身的结果）

这就是**官方评测会看到的数字**——模型在 6 个评测设置上的三项指标。
表 1 与图 A / 图 B / 图 C 全部属于这一块。

### 区块二：参考对照（不是评估标准）

两个**人造系统**，用来回答「这个分数算好吗」：

| 对照系统 | 它是什么 | 用来回答什么问题 |
|---|---|---|
| **随机初始化** | 同一个模型、同一份数据，但**没训练过** | 「模型到底学到了什么」——训练后必须显著优于它，否则等于没训 |
| **type-level oracle** | 给每个目标赋它自己的**真实均值**，完全不看语境 | 「天花板在哪」——它在指标 1 / RMSE 上能拿满分，但指标 2 必然是 `nan`（组内 std 恒为 0）。我们的模型如果指标 2 也接近 nan，说明还没真正做到「因句而异」 |

**这两个都不是提交系统的成绩**，所以它们被单独放在区块二，
不与评估标准混在同一张表、同一根柱子里。

**三张主图各管一件事**：

| 图 | 内容 | 看什么 |
|---|---|---|
| 图 A | 预测 vs 真值散点（每个「子集×列」一个面板） | 点是否落在对角线上；ρ 是整体排序能力，RMSE 是数值校准 |
| 图 B | 6 个评测设置 × 三项指标 | **评估标准**本身：哪个设置强、哪个设置弱、指标 2 有没有塌成 nan |
| 图 C | 组内 std 对照散点（人工 std vs 预测 std） | 这就是指标 2 的可视化：横轴是金标准，纵轴是系统输出，ρ 即指标 2 |

**为什么图 C 最关键**：指标 2 直接决定是否出局。
如果图 C 的点挤成一条水平线（预测的组内 std 几乎不变），
说明模型没有真正对语境敏感——哪怕指标 1 很高，这份提交也会在指标 2 上拿 `nan`。

In [ ]:



def style_axes(ax):
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)
    ax.grid(True, alpha=0.22, linewidth=0.6)
    ax.set_axisbelow(True)


def annotate_box(ax, x, y, text, color="#2C3E50", ha="left", va="top"):
    ax.text(x, y, text, ha=ha, va=va, fontsize=8.4, color=color,
            bbox=dict(boxstyle="round,pad=0.32", facecolor="white",
                      edgecolor=color, linewidth=0.8, alpha=1.0))


def label_bars(ax, items, ylim, fontsize=7.0):
    lo, hi = ylim
    gap = (hi - lo) * 0.058
    placed = []
    for x, y, t in sorted(items, key=lambda z: (z[0], -z[1])):
        yy = y
        for px, py in placed:
            if abs(px - x) < 0.32 and abs(py - yy) < gap:
                yy = py + gap
        placed.append((x, yy))
        va = "bottom" if y >= 0 else "top"
        ax.text(x, yy, t, ha="center", va=va, fontsize=fontsize, color="#2C3E50",
                bbox=dict(boxstyle="round,pad=0.16", facecolor="white",
                          edgecolor="none", alpha=1.0))


BLUE, ORANGE, GREY, GREEN, RED = "#3C6E9E", "#C2703D", "#8A8F98", "#2E7D5B", "#B03A2E"


MAIN_RUN = next((r for r in ALL_RUNS if r["mode"] == "joint"), ALL_RUNS[-1])
print("主图使用的运行:", MAIN_RUN["run_name"], "| 评测设置:", len(MAIN_RUN["eval_cols"]))

In [ ]:



_rows = []
for _r in ALL_RUNS:
    for _n, _cn, _j in _r["eval_cols"]:
        _rows.append({
            "运行": _r["run_name"],
            "子集": _n,
            "标签列": _cn,
            "指标1 ρ": _r["metrics"].get(f"{_n}|{_cn}_指标1_ρ", float("nan")),
            "指标2 ρ": _r["metrics"].get(f"{_n}|{_cn}_指标2_ρ", float("nan")),
            "指标3 RMSE": _r["metrics"].get(f"{_n}|{_cn}_指标3_RMSE", float("nan")),
            "组内std退化目标数": _r["metrics"].get(f"{_n}|{_cn}_组内std退化目标数", 0),
            "样例数": _r["metrics"].get(f"{_n}|{_cn}_样例数", 0),
        })
RESULTS = pd.DataFrame(_rows)
MAIN_RESULTS = RESULTS[RESULTS["运行"] == MAIN_RUN["run_name"]].reset_index(drop=True)
print("=" * 96)
print(f"表 1  评估标准 —— {MAIN_RUN['run_name']}（联合运行覆盖全部评测设置）")
print("=" * 96)
print(MAIN_RESULTS.round(4).to_string(index=False))
print(f"\n合计（各评测设置取平均）：指标1 ρ={MAIN_RESULTS['指标1 ρ'].mean():.4f} | "
      f"指标2 ρ={MAIN_RESULTS['指标2 ρ'].mean():.4f} | "
      f"RMSE={MAIN_RESULTS['指标3 RMSE'].mean():.4f}")
print("→ 指标 2 若是 nan，说明该设置下模型对同一目标的所有例句给出了相同分数，"
      "会在官方评测中直接出局。")


_summary_dir = OUTPUT_ROOT / "summary"
_summary_dir.mkdir(parents=True, exist_ok=True)
RESULTS.to_csv(_summary_dir / "results_all_runs.csv", index=False)
print("\n已保存:", _summary_dir / "results_all_runs.csv")







_MUST_MATCH = ["optim", "fit_tier", "trainable_scope", "mixed_precision", "lr", "epochs",
               "max_length", "weight_decay", "warmup_ratio", "huber_delta", "lambda_rank",
               "mu_std", "seed", "n_outputs", "effective_batch"]
_MAY_DIFFER = {
    "model_key": "模型身份",
    "model_id": "模型身份",
    "pooling": "架构决定的池化方式（mean / last）",
    "micro_batch": "batch 分解方式（乘积与 effective_batch 相同即等价）",
    "grad_accum": "batch 分解方式（乘积与 effective_batch 相同即等价）",
    "gradient_checkpointing": "只影响速度与显存，数学上等价（激活重算不改数值）",
    "gpu_index": "用哪一张**同型号**卡（只影响并行调度，不影响数值）",
    "train_subset": "本次运行的子集（single vs joint 本就不同）",
}
print("\n" + "=" * 78)
print("本次运行的完整口径")
print("=" * 78)
_s = ALL_RUNS[0]["settings"]
print(f"\n【必须与另一条基线逐字相同】{len(_MUST_MATCH)} 项")
for _k in _MUST_MATCH:
    print(f"  {_k:24s} = {_s.get(_k)}")
print(f"\n【允许不同（各有明确理由，不影响可比性）】{len(_MAY_DIFFER)} 项")
for _k, _why in _MAY_DIFFER.items():
    print(f"  {_k:24s} = {_s.get(_k)}      ← {_why}")
print("\n口径文件:", _summary_dir / f"run_settings_{ALL_RUNS[0]['run_name']}.json")
print("⚠️ 跑完两条基线后，请 diff 两边的 run_settings_*.json：")
print(f"   「必须逐字相同」的 {len(_MUST_MATCH)} 项若有任何一项不同 → 对照结论被污染，需把两边改成一致后重跑。")
print("   尤其注意 fit_tier / optim：档位由「两条基线中较大者的参数量 + 本机显存」统一决定，"
      "同一台机器上必然相同；若不同，说明两边跑的**不是同一台机器**。")

In [ ]:



_panels = MAIN_RUN["eval_cols"]
_pred = MAIN_RUN["preds"]
_gold = MAIN_RUN["gold"]
_groups = MAIN_RUN["groups"]
_subsets = MAIN_RUN["subset_ids"]
_nc = min(3, len(_panels))
_nr = math.ceil(len(_panels) / _nc)
fig, axes = plt.subplots(_nr, _nc, figsize=(4.8 * _nc, 4.5 * _nr), squeeze=False)
for _k, (_name, _cname, _j) in enumerate(_panels):
    ax = axes[_k // _nc][_k % _nc]
    _m = _subsets == _name
    ax.scatter(_gold[_m, _j], _pred[_m, _j], s=14, alpha=0.42, color=BLUE, edgecolors="none")
    _lo = max(0.0, float(min(_gold[_m, _j].min(), _pred[_m, _j].min())) - 0.3)
    _hi = float(max(_gold[_m, _j].max(), _pred[_m, _j].max())) + 0.3
    ax.plot([_lo, _hi], [_lo, _hi], "--", color=GREY, linewidth=1.1, zorder=1)
    ax.set_xlim(_lo, _hi); ax.set_ylim(_lo, _hi)
    ax.set_xlabel(T("gold")); ax.set_ylabel(T("pred"))
    ax.set_title(f"{_name}·{_cname}", fontsize=10.5)
    _r = stats.spearmanr(_pred[_m, _j], _gold[_m, _j])[0]
    _e = float(np.sqrt(((_pred[_m, _j] - _gold[_m, _j]) ** 2).mean()))
    annotate_box(ax, _lo + 0.04 * (_hi - _lo), _hi - 0.04 * (_hi - _lo),
                 f"ρ = {_r:.3f}\nRMSE = {_e:.3f}")
    style_axes(ax)
for _k in range(len(_panels), _nr * _nc):
    axes[_k // _nc][_k % _nc].axis("off")
fig.suptitle(f"{T('figA')} ({MAIN_RUN['run_name']})", fontsize=12.5,
             fontweight="bold", y=1.02)
fig.tight_layout()
_figA = _summary_dir / "figA_pred_vs_gold.png"
fig.savefig(_figA, dpi=160, bbox_inches="tight")
plt.show()
plt.close(fig)
print("已保存:", _figA)

In [ ]:



_metric_defs = [("指标1_ρ", T("rho_up"), T("m1")),
                ("指标2_ρ", T("rho_up"), T("m2")),
                ("指标3_RMSE", T("rmse_down"), T("m3"))]
_xlabels = [f"{n}\n{c}" for n, c, _j in _panels]

fig, axes = plt.subplots(1, 3, figsize=(2.0 * len(_panels) + 3.6, 5.2))
for _mi, (_suffix, _ylabel, _mtitle) in enumerate(_metric_defs):
    ax = axes[_mi]
    _vals = [MAIN_RUN["metrics"].get(f"{n}|{c}_{_suffix}", float("nan")) for n, c, _j in _panels]
    _finite = [v for v in _vals if np.isfinite(v)]
    _heights = [v if np.isfinite(v) else 0.0 for v in _vals]
    _colors = [BLUE if np.isfinite(v) else "#D8DBE0" for v in _vals]
    ax.bar(np.arange(len(_panels)), _heights, 0.6, color=_colors, edgecolor="white")


    _lo = min([0.0] + _finite) if _finite else 0.0
    _hi = max([0.0] + _finite) if _finite else 1.0
    _span = max(_hi - _lo, 1e-6)
    ax.set_ylim(_lo - 0.08 * _span, _hi + 0.22 * _span)
    _items = [(k, v, f"{v:.3f}") for k, v in enumerate(_vals) if np.isfinite(v)]
    label_bars(ax, _items, ax.get_ylim(), fontsize=7.4)
    for k, v in enumerate(_vals):
        if not np.isfinite(v):

            ax.text(k, _lo + 0.02 * _span, "nan", ha="center", va="bottom",
                    fontsize=8.0, color=RED, fontweight="bold")
    ax.axhline(0, color="#B0B4BA", linewidth=0.8, zorder=0)
    ax.set_xticks(np.arange(len(_panels)))
    ax.set_xticklabels(_xlabels, fontsize=7.2)
    ax.set_ylabel(_ylabel)
    ax.set_title(f"({chr(97 + _mi)}) {_mtitle}", fontsize=10.5)
    style_axes(ax)
fig.suptitle(f"{T('figB')}: {MAIN_RUN['run_name']} on {len(_panels)} {T('n_settings')}",
             fontsize=12.5, fontweight="bold", y=1.02)
fig.tight_layout()
_figB = _summary_dir / "figB_evaluation_standard.png"
fig.savefig(_figB, dpi=160, bbox_inches="tight")
plt.show()
plt.close(fig)
print("已保存:", _figB)

In [ ]:







_c_sub = list(_panels)
_nc = min(3, len(_c_sub))
_nr = math.ceil(len(_c_sub) / _nc)
fig, axes = plt.subplots(_nr, _nc, figsize=(4.6 * _nc, 4.3 * _nr), squeeze=False)
for _k, (_name, _cname, _j) in enumerate(_c_sub):
    ax = axes[_k // _nc][_k % _nc]
    _m = _subsets == _name
    _gs = pd.DataFrame({"g": _groups[_m], "p": _pred[_m, _j], "y": _gold[_m, _j]}).groupby("g")
    _ps = _gs["p"].std(ddof=1)
    _ys = _gs["y"].std(ddof=1)
    _ok = _ps.notna() & _ys.notna()
    _r2 = stats.spearmanr(_ps[_ok], _ys[_ok])[0] if _ok.sum() > 2 else float("nan")
    ax.scatter(_ys[_ok], _ps[_ok], s=20, alpha=0.55, color=ORANGE, edgecolors="none")
    _mx = float(max(_ys[_ok].max(), _ps[_ok].max())) * 1.10 if _ok.sum() else 1.0
    ax.plot([0, _mx], [0, _mx], "--", color=GREY, linewidth=1.1, zorder=1)
    ax.set_xlim(0, _mx); ax.set_ylim(0, _mx)
    ax.set_xlabel(T("gold_std"))
    ax.set_ylabel(T("pred_std"))
    ax.set_title(f"{_name}·{_cname}", fontsize=10.5)
    annotate_box(ax, _mx * 0.03, _mx * 0.97,
                 f"{T('m2rho')} {_r2:.3f}\n{T('n_targets')} {int(_ok.sum())}\n"
                 f"{T('const_std')} {int((_ps[_ok] == 0).sum())}")
    style_axes(ax)
for _k in range(len(_c_sub), _nr * _nc):
    axes[_k // _nc][_k % _nc].axis("off")
fig.suptitle(f"{T('figC')}: {T('gold_axis')} ({MAIN_RUN['run_name']})",
             fontsize=12.5, fontweight="bold", y=1.02)
fig.tight_layout()
_figC = _summary_dir / "figC_group_std.png"
fig.savefig(_figC, dpi=160, bbox_inches="tight")
plt.show()
plt.close(fig)
print("已保存:", _figC)

In [ ]:




_run_names = [r["run_name"] for r in ALL_RUNS]
_run_labels = []
for _r in ALL_RUNS:
    if _r["mode"] == "joint":
        _run_labels.append(T("joint_short"))
    else:
        _run_labels.append(T("single_short", short=short_of(_r["subset"])))

fig, axes = plt.subplots(1, 3, figsize=(2.1 * len(ALL_RUNS) + 3.2, 4.8))
for _mi, (_suffix, _ylabel, _mtitle) in enumerate(_metric_defs):
    ax = axes[_mi]
    _vals = [_r["metrics"].get(f"合计|_{_suffix}", float("nan")) for _r in ALL_RUNS]
    _finite = [v for v in _vals if np.isfinite(v)]
    _heights = [v if np.isfinite(v) else 0.0 for v in _vals]
    _colors = [ORANGE if _r["mode"] == "joint" else BLUE for _r in ALL_RUNS]
    ax.bar(np.arange(len(ALL_RUNS)), _heights, 0.62, color=_colors, edgecolor="white")
    _lo = min([0.0] + _finite) if _finite else 0.0
    _hi = max([0.0] + _finite) if _finite else 1.0
    _span = max(_hi - _lo, 1e-6)
    ax.set_ylim(_lo - 0.08 * _span, _hi + 0.24 * _span)
    label_bars(ax, [(k, v, f"{v:.3f}") for k, v in enumerate(_vals) if np.isfinite(v)],
               ax.get_ylim(), fontsize=7.6)
    for k, v in enumerate(_vals):
        if not np.isfinite(v):
            ax.text(k, _lo + 0.02 * _span, "nan", ha="center", va="bottom",
                    fontsize=8.0, color=RED, fontweight="bold")
    ax.axhline(0, color="#B0B4BA", linewidth=0.8, zorder=0)
    ax.set_xticks(np.arange(len(ALL_RUNS)))
    ax.set_xticklabels(_run_labels, fontsize=7.4)
    ax.set_ylabel(_ylabel)
    ax.set_title(f"({chr(97 + _mi)}) {_mtitle}", fontsize=10.5)
    style_axes(ax)
_handles = [plt.Rectangle((0, 0), 1, 1, color=BLUE), plt.Rectangle((0, 0), 1, 1, color=ORANGE)]
fig.legend(_handles, [T("single"), T("joint")],
           loc="upper center", bbox_to_anchor=(0.5, 0.99), ncol=2, fontsize=9, frameon=False)
fig.suptitle(T("figD"), fontsize=12.5, fontweight="bold", y=1.06)
fig.tight_layout()
_figD = _summary_dir / "figD_run_comparison.png"
fig.savefig(_figD, dpi=160, bbox_inches="tight")
plt.show()
plt.close(fig)
print("已保存:", _figD)

In [ ]:






print("=" * 96)
print("区块二  参考对照（不是评估标准，仅用于解读上面的数字）")
print("=" * 96)
_cmp_rows = []
for _r in ALL_RUNS:
    for _sys, _m in [("训练后模型", _r["metrics"]),
                     ("随机初始化", _r["random_metrics"]),
                     ("type-level oracle", _r["oracle_metrics"])]:
        _cmp_rows.append({
            "运行": _r["run_name"], "对照系统": _sys,
            "指标1 ρ": _m.get("合计|_指标1_ρ", float("nan")),
            "指标2 ρ": _m.get("合计|_指标2_ρ", float("nan")),
            "指标3 RMSE": _m.get("合计|_指标3_RMSE", float("nan")),
        })
COMPARE = pd.DataFrame(_cmp_rows)
print(COMPARE.round(4).to_string(index=False))
COMPARE.to_csv(_summary_dir / "reference_comparison.csv", index=False)

print("\n--- 三个系统分别是什么 ---")
print("随机初始化      ：同一模型、同一份数据，但一步都没训练。"
      "它是「瞎猜」的下界；训练后模型必须显著高于它，否则等于没训。")
print("type-level oracle：给每个目标赋它自己的真实均值（不看语境）。"
      "它在指标 1 / RMSE 上接近满分，但指标 2 必然是 nan"
      "（同一目标的所有例句拿到同一个分数 → 组内 std 恒为 0 → 相关系数未定义）。")
print("→ 我们的模型如果指标 2 也接近 nan，说明还没有真正做到「因句而异」。")

In [ ]:


_cmp_systems = [(T("trained"), "metrics", BLUE),
                (T("random_init"), "random_metrics", GREY),
                (T("oracle"), "oracle_metrics", ORANGE)]
fig, axes = plt.subplots(1, 3, figsize=(11.0, 4.6))
for _mi, (_suffix, _ylabel, _mtitle) in enumerate(_metric_defs):
    ax = axes[_mi]
    _items = []
    for _si, (_sys, _key, _c) in enumerate(_cmp_systems):
        _v = MAIN_RUN[_key].get(f"合计|_{_suffix}", float("nan"))
        _x = _si
        if np.isfinite(_v):
            ax.bar(_x, _v, 0.55, color=_c, edgecolor="white")
            _items.append((_x, _v, f"{_v:.3f}"))
        else:
            ax.bar(_x, 0.0, 0.55, color="#D8DBE0", edgecolor="white")
    _finite = [y for _x, y, _t in _items]
    _lo = min([0.0] + _finite) if _finite else 0.0
    _hi = max([0.0] + _finite) if _finite else 1.0
    _span = max(_hi - _lo, 1e-6)
    ax.set_ylim(_lo - 0.08 * _span, _hi + 0.24 * _span)
    label_bars(ax, _items, ax.get_ylim(), fontsize=8.4)

    if _suffix == "指标2_ρ":
        ax.text(2, _lo + 0.02 * _span, "nan", ha="center", va="bottom",
                fontsize=9.0, color=RED, fontweight="bold")
    ax.axhline(0, color="#B0B4BA", linewidth=0.8, zorder=0)
    ax.set_xticks(range(len(_cmp_systems)))
    ax.set_xticklabels([s for s, _k, _c in _cmp_systems], fontsize=8.2)
    ax.set_ylabel(_ylabel)
    ax.set_title(f"({chr(97 + _mi)}) {_mtitle}", fontsize=10.5)
    style_axes(ax)
fig.suptitle(f"{T('figE')} ({T('not_standard')}): {MAIN_RUN['run_name']}",
             fontsize=12.0, fontweight="bold", y=1.03)
fig.tight_layout()
_figE = _summary_dir / "figE_reference_comparison.png"
fig.savefig(_figE, dpi=160, bbox_inches="tight")
plt.show()
plt.close(fig)
print("已保存:", _figE)

## 附录：推理与提交（`*-pred.tsv` 导出）

**为什么把导出放在最后**：提交格式是本任务**最容易在最后一天翻车**的环节。
下面这份导出函数已经把所有已知的坑写成了断言——任何一条不满足都直接 `raise`，
而不是让它静默走到提交那天。

**格式规范（官方）**：

- 提交物是**一个 zip**，内含 4 个文件：`en-nn-pred.tsv` / `en-pv-pred.tsv` /
  `de-nn-pred.tsv` / `de-pv-pred.tsv`
- **无表头**，制表符分隔
- 名词复合词每行 **3** 列：`ContextID` ⇥ `修饰语预测` ⇥ `中心语预测`（**修饰语在前**）
- 小品词动词每行 **2** 列：`ContextID` ⇥ `单一预测`
- 测试集**不含** `*Avg` / `*Std` 列，载入逻辑必须容忍缺列
- 预测值必须落在 **[0, 5]**；输出**不得是常数**（否则指标 2 = nan）

In [ ]:

def export_pred(tsv_path, preds, lang, task):
    test = pd.read_csv(tsv_path, sep="\t")
    ids = test["ContextID"].astype(str)
    preds = np.asarray(preds, dtype=float)
    if preds.ndim == 1:
        preds = preds.reshape(-1, 1)

    if task == "nn":
        assert preds.shape[1] == 2, f"名词复合词需要 2 列（修饰语、中心语），收到 {preds.shape[1]}"
        body = pd.DataFrame({"ContextID": ids, "mod": preds[:, 0], "head": preds[:, 1]})
    else:
        assert preds.shape[1] == 1, f"小品词动词需要 1 列，收到 {preds.shape[1]}"
        body = pd.DataFrame({"ContextID": ids, "pred": preds[:, 0]})

    assert len(body) == len(test), "行数与测试集不一致"
    assert set(body["ContextID"]) == set(ids), "ContextID 集合与测试集不一致"
    _v = body.select_dtypes("number").to_numpy()
    assert np.isfinite(_v).all(), "预测里出现 NaN/inf"
    assert _v.min() >= 0.0 and _v.max() <= 5.0, f"预测超出 [0,5]：[{_v.min():.3f}, {_v.max():.3f}]"

    out = OUTPUT_ROOT / f"{lang}-{task}-pred.tsv"
    body.to_csv(out, sep="\t", index=False, header=False)
    return out






_exported = []
for _r in ALL_RUNS:
    for _n, _cn, _j in _r["eval_cols"]:
        if _cn != SUBSET_SPECS[_n]["cols"][0]:
            continue
        _sp = SUBSET_SPECS[_n]
        _m = _r["subset_ids"] == _n
        _cols = [_r["preds"][_m, jj] for jj in range(len(_sp["cols"]))]
        _tsv = OUTPUT_ROOT / f"_demo_test_{_n}.tsv"
        pd.DataFrame({"ContextID": [f"{i:05d}" for i in range(int(_m.sum()))],
                      "Context": _r["val_df"].loc[_m, "Context"].to_numpy()
                      }).to_csv(_tsv, sep="\t", index=False)
        _out = export_pred(_tsv, np.stack(_cols, axis=1), _sp["lang"], _sp["task"])
        _exported.append((_r["run_name"], _n, _out))
        print(f"[{_r['run_name']} / {_n}] -> {_out.name}")
print("\n→ 无表头、列数正确、值域 [0,5]、ContextID 原样透传。")
print("→ 正式提交时把 4 个 *-train.tsv 换成官方测试 TSV，文件名保持 en-nn-pred.tsv 等 4 个。")

## 结论与延伸

### 这份基线的运行前置条件

> 本文件是 **Qwen2.5-1.5B-Instruct** 的专属基线。另一条基线（DeBERTa / Qwen）用**完全相同**的
> 数据处理、超参、指标与绘图代码，只换 `MODEL_KEY` 一个变量，可直接对照。

| 项 | 要求 | 说明 |
|---|---|---|
| Python | ≥ 3.10 | 本机实测 3.12.7 |
| torch | ≥ 2.4 | 本机实测 2.14.0 |
| transformers | ≥ 5.0 | **5.x 的 API 与 4.x 不同**：`eval_strategy`、`processing_class`、`warmup_steps`、`dtype=` 都是 5.x 写法；代码已做双兼容（签名探测自动切换） |
| datasets | ≥ 3.0 | 本机实测 3.6.0 |
| 网络 | 云端 GPU 可直连 `huggingface.co`；本机通常不通 | Step1 会**自动探测**官方源，不通就换 `hf-mirror.com`，下载时再兜一层重试。也可设 `COMPARTMENT_HF_MODE=official\|mirror\|cache` 强制指定 |
| **显存** | **单卡 ≥ 12GiB 可用**（T4 14.56GiB 即可） | 训练「档位」按 **参考参数量 1.55e9**（两条基线中较大者）+ 本机显存自动选：t0 fp32 AdamW(16 B/param) → t1 8bit AdamW(10) → t2 Adafactor(8) → t3 只训顶部 50% 层(10)。T4 上落在 t2，**仍是全参微调**。Step8 会用真实模型真跑 2 步验证，OOM 自动降一档 |
| 内存 | ≥ 16GB | 主干 fp32 权重约 3.6GB / 6.2GB |
| 磁盘 | ≥ 10GB 可用 | 5 次运行只保留**联合运行**的权重（~3.6GB），训练用 checkpoint 训完即删 |
| 时间 | 5 次运行（4 个单子集 + 1 个联合） | 联合运行的数据量是单子集的 4 倍，总时长约为「单子集总时长 × 2」；Step8 开跑前会打印**时长投影**，超过 11 小时会报警 |

### 显存够不够：先看这张表

全参微调 + AdamW 的**固定开销**（权重 fp32 4 B/param + 梯度 fp32 4 B/param
+ AdamW 的 m/v 8 B/param = **16 B/param**，再加上开混合精度时 AMP 的 fp16 权重副本 2 B/param
→ **18 B/param**）**与 batch 大小无关**，所以「降 batch / 加梯度累积」救不了它。
真正能降下来的只有三样：**优化器状态**、**AMP 的 fp16 副本**、**可训练参数的范围**。

| 模型 | 参数量 | fp32 AdamW+AMP | 8bit+AMP | Adafactor+AMP | Adafactor 无 AMP（t3） |
|---|---|---|---|---|---|
| `microsoft/deberta-v2-xlarge` | 885M | 14.8 GiB | 9.9 GiB | 8.4 GiB | 6.8 GiB ✓ |
| `Qwen/Qwen2.5-1.5B-Instruct` | 1544M | **25.9 GiB** | 17.3 GiB | 14.7 GiB ✗ | **11.8 GiB ✓** |

（表中为「固定开销」，不含激活；实测 DeBERTa 在 t2 档的**峰值 9.93 GiB**，
比固定开销多出的 ~1.5 GiB 就是激活 + 碎片 + CUDA context。）

**T4 只有 14.56 GiB**，所以在 T4 上能保住「全参微调」语义的档位是
**t3（Adafactor + 关 AMP，8.2 B/param ≈ 11.8 GiB）** —— 没有任何参数被冻结。

### 为什么「9B 都能全量微调，1.5B 却超了」

这两句话说的**不是同一件事**，差别在三个地方：

| | 9B 全参微调（业界常见） | 本项目 1.5B 全参微调 |
|---|---|---|
| 硬件 | 2 × A100 **80GB**（合计 160GB） | 2 × T4 **14.56GB**（合计 29.1GB） |
| 固定开销 | 9B × 18 B = **162GB**，靠 ZeRO-3 分片塞进 160GB（还要再省） | 1.5B × 18 B = **25.9GB**，单卡 14.56GB 装不下 |
| 是否分片 | 用 ZeRO-3 / FSDP 把参数切到多卡 | notebook 是**单进程**，每卡都要放完整一份 |

关键在于：**DDP（数据并行）不省显存** —— 每张卡仍要放一整份模型+梯度+优化器状态，
只是把数据分半（提速用）。只有 **ZeRO-3 / FSDP 参数分片**才会把 25.9GB 摊到两张卡
（13GB/卡），而那需要 `accelerate launch --num_processes 2`，
**在 papermill 的单进程里跑不了**；而且本 notebook 自定义了分组采样器
（让同一目标的例句落在同一个 batch），DDP 下若不改成分布式采样器，
两个进程会取到**同一批样本** → 训练结果悄悄错掉。

所以本 notebook 的选择是：**单卡 + 把固定开销从 18 B/param 降到 8.2 B/param
（Adafactor + 关 AMP）**，这样 1.5B 也能在 T4 上做**完整的全参微调**。

### 训练档位（tier）：显存不够时自动降级，但两条基线绝不分叉

**显存模型（云端实跑校准过，务必照抄这份口径）**：

```
每参数开销 = 权重 fp32 4
           + （开 AMP 时）fp16 权重副本 2   ← 反向传播要用它算 grad_input，
                                               被 autograd 存到 backward 结束
           + 梯度 fp32 4 × 可训比例
           + 优化器状态 × 可训比例   AdamW 8 / 8bit AdamW 2 / Adafactor ≈0.2
```

> 实测校验：DeBERTa-xlarge（885M）在 t2 档的**实测峰值 9.93 GiB**，
> 按上式算 10.2 B/param × 885M = 8.4 GiB + 激活/碎片 ≈ 1.5 GiB → 9.9 GiB ✓ 对得上。
> 第一版估算漏了 AMP 的 2 B/param → Qwen-1.5B 少算 3.1 GiB → 以为装得下 → 实测 bs=1 也 OOM。

| 档位 | 含义 | optim | AMP | 可训层 | 固定开销 | 什么时候选它 |
|---|---|---|---|---|---|---|
| t0 | 全参微调 · fp32 AdamW | `adamw_torch` | 开 | 100% | 18 B/param | 40GB+ 的卡（A100 等） |
| t1 | 全参微调 · 8bit AdamW | `adamw_bnb_8bit` | 开 | 100% | 12 B/param | 装了 bitsandbytes 且 ≥24GB 卡 |
| t2 | 全参微调 · Adafactor | `adafactor` | 开 | 100% | 10.2 B/param | 16GB+ 的卡（L4 24GB 会选 t1） |
| **t3** | **全参微调 · Adafactor · 关 AMP** | `adafactor` | 关 | 100% | **8.2 B/param** | **T4 14.56GB 就是这一档**（仍全参微调，只是不用混合精度） |
| t4 | 顶部 50% 层 · Adafactor | `adafactor` | 开 | 50% | 8.1 B/param | 想要 AMP 速度、可以接受冻一半层时 |
| t5 | 顶部 50% 层 · Adafactor | `adafactor` | 关 | 50% | 6.1 B/param | 更省显存 |
| t6 | 顶部 25% 层 · Adafactor | `adafactor` | 关 | 25% | 5.05 B/param | 兜底 |

> ⚠️ **上表的前提是「单卡模式」**。若进程能看见 2 张卡，transformers 的 Trainer 会自动套
> `nn.DataParallel`，其反向传播要**再复制一整份梯度**（Qwen-1.5B 多 5.75 GiB）→ 必然 OOM。
> Step1 用 `CUDA_VISIBLE_DEVICES` 把进程限制成单卡，`build_training_args` 里再钉 `args._n_gpu = 1` 兜底。

**选档规则只有一条**：用**两条基线中较大者的参数量**（`FIT_REF_PARAMS = 1.55e9`）作参考，
配合本机显存选第一个装得下的档位（判据用「micro-batch=1 的最小激活」，
因为激活是 batch 的事、由探测阶段逐档降）。
这样同一台机器上两条基线**必然落在同一档** → 对照口径不会被显存差异污染。
若估算仍偏乐观，Step8 会用**真实模型真跑 2 步**验证，OOM 就自动降一档重来；
全部档位都失败时给出精确诊断（而不是一句 out of memory）。

> t3 档为什么「关 AMP」：开 AMP 时 PyTorch 会把权重的 fp16 副本留到反向传播用，
> 多占 2 B/param。Qwen-1.5B 正是差这 2 B/param（3.1 GiB）而装不下 14.56 GiB 的卡。
> 关掉 AMP 就省出这 3.1 GiB，代价是纯 fp32 训练慢一些（T4 没有 TF32）——
> 但 T4 上 batch 很小、属于「发射受限」而非「算力受限」，实测慢得不多。

### 「两张 T4 能不能凑成 30G 用？」——不能，而且云端实跑发现：**两张卡反而是致命的**

**结论先说**：CUDA 显存**不能跨卡合并**。一个张量只能待在一张卡上，
「2 × 14.56 GiB」不会变成「一张 30 GiB 的卡」。

**而且本项目在云端实跑里发现了一件更糟的事**：只要进程能看见 2 张卡，
transformers 的 Trainer 就会**自动**把模型套上 `nn.DataParallel`：

```python
# transformers/trainer.py :: _wrap_model()
if self.args.n_gpu > 1 and not is_loaded_in_8bit and not is_loaded_in_4bit:
    model = nn.DataParallel(model)
```

`nn.DataParallel` 的反向传播要经过 `Broadcast` → `ReduceAddCoalesced`
→ `comm.reduce_add_coalesced(...)`，**额外分配并复制一整份梯度** →
**梯度显存直接翻倍**。Qwen-1.5B 就是多 5.75 GiB → 14.56 GiB 的卡必然 OOM。

云端日志里的证据非常干净：**t3 / t4 / t5 三个档位 × 8/4/2/1 四个 batch 全部 OOM**，
每次都死在同一个地方：

```
Tried to allocate 54.00 MiB ... Of the allocated memory 14.26 GiB is allocated by PyTorch
  堆栈: torch/nn/parallel/_functions.py → ReduceAddCoalesced → nccl.reduce
```

（14.26 GiB ≈ 权重 5.75 + 梯度 5.75 + 正在复制的第二份梯度 2.76 ✓ 数字闭环）

所以「用两张卡」在单进程里不但没省显存，反而**多花了 5.75 GiB**。
本项目的做法是：**用 `CUDA_VISIBLE_DEVICES` 把进程限制成只看见 1 张卡**
（Step1 已设，`build_training_args` 里再钉一次 `args._n_gpu = 1` 兜底），
让 Trainer 走单卡路径，梯度显存不再翻倍。

| 想做的事 | 能不能把两张卡的显存合起来用 | 说明 |
|---|---|---|
| **DDP（数据并行）** | **不能** | 每张卡各存**一整份**模型+梯度+优化器状态，只是把 batch 分半 → 提速，不省显存 |
| **nn.DataParallel（Trainer 自动套的）** | **不能，而且更糟** | 反向传播还要**再复制一份梯度** → 显存 ×2 |
| **ZeRO-3 / FSDP（参数分片）** | **能** | 每卡只放 1/2 → 这才是「29 GiB 一起用」 |
| **两条基线各占一张卡** | 不需要合并 | 两个独立实验各跑各的，墙钟时间直接减半 |

**为什么本 notebook 不直接上 ZeRO-3/FSDP**（三个实打实的原因）：

1. 需要**多进程**启动：`accelerate launch --num_processes 2` 或 `torchrun`。
   papermill / `jupyter nbconvert` 起的是**普通单进程**，不会自动开 DDP。
2. 本 notebook 自定义了 `TargetGroupedSampler`（让同一目标的例句落在同一个 batch，
   否则组内排序损失与组内 std 损失形同虚设）。DDP 下若不改成分布式采样器，
   **两个进程会取到同一批样本** → 训练结果悄悄错掉，而且**不报错**。
3. 改造要动的地方多（采样器、Trainer 子类、跨进程结果汇总），属于大改，
   而当前阻塞点（Qwen 装不下）用单卡 + 档位机制已经能解。

**所以现在用的是最省事的方案**：

```bash
COMPARTMENT_GPU=0   # DeBERTa 版（单卡，梯度显存不翻倍）
COMPARTMENT_GPU=1   # Qwen 版（同一台机器上跑同一个档位 → 口径一致）
```

两条基线各占一张卡同时跑，**两张卡的算力都用满了**，墙钟时间减半；
而「29 GiB 合成一张卡」这件事，等以后真的要训 7B/9B 时再用 FSDP 脚本形态来做。

### 多卡怎么用：先分清「提速」和「省显存」

| 用法 | 每卡上放什么 | 能降低单卡显存需求吗 | 能提速吗 |
|---|---|---|---|
| 单进程 | 全部在一张卡 | — | 基准 |
| **DDP（数据并行）** | **每卡各存一整份**模型+梯度+优化器状态，只是数据分半 | **不能**（每卡仍是完整那一份） | 能（约 1.8×） |
| **ZeRO-3 / FSDP（参数分片）** | 参数/梯度/优化器状态**切分**到多卡 | **能**（这才是用满多卡显存的办法） | 能 |
| 两条基线各占一张卡 | 两条基线是**互不相干的实验**，各占一卡同时跑 | 不需要（各跑各的） | 能（墙钟减半） |

**对 Qwen2.5-1.5B 具体意味着什么**：DDP **救不了**它 —— 每卡仍要装 23.0 GiB 的固定开销。
本 notebook 的做法是走 **t3 档**（全参微调 + Adafactor + 关 AMP = 8.2 B/param ≈ 11.8 GiB，单卡就够）；
真想用满 29 GiB 训更大的模型，则要上 ZeRO-3/FSDP 脚本形态（见上一节的三条理由）。

**注意**：papermill / `jupyter nbconvert` 起的是**普通单进程**，**不会自动开 DDP**。
真要用 DDP 得显式 `accelerate launch --num_processes 2 notebook.ipynb`，
而且必须把自定义采样器改成分布式采样器，否则两个进程会取到同一批样本。

### 训练时间够不够：开跑前就有投影

Step8 探测时会量出「每样本耗时」，据此投影 **5 次运行**（4 个单子集 + 1 个联合）
的总时长并打印；若超过 `TIME_BUDGET_H`（默认 11 小时，给评估与出图留 1 小时余量），
会给出三条可选做法。**它只报警、不自动改 EPOCHS** —— 自动改会让两条基线的
训练轮数不同，对照口径就废了。


### 图上的中文：云端可能没有中文字体

Kaggle 等云端镜像的默认字体（DejaVu Sans）**不含中文字形**，中文会静默变成方块「口口口」。
本 notebook 的处理是：先找系统字体 → 找不到就下载 Noto Sans CJK → 仍失败则**图注自动切英文**。
也可以用环境变量强制：

```bash
export COMPARTMENT_NO_CJK=1     # 强制英文图注（不下载 16MB 字体）
```

### 自动跑了哪些运行

`RUN_PLAN`（Step1）是唯一的开关：

| 运行 | 训练数据 | 覆盖的评测设置 | 用途 |
|---|---|---|---|
| `single / en-nn` | 只有 en-nn | en-nn·ModAvg、en-nn·HeadAvg | 单子集分析 |
| `single / de-nn` | 只有 de-nn | de-nn·ModAvg、de-nn·HeadAvg | 单子集分析 |
| `single / en-pv` | 只有 en-pv | en-pv·Avg | 单子集分析 |
| `single / de-pv` | 只有 de-pv | de-pv·Avg | 单子集分析 |
| `joint` | 四个子集拼起来 | 全部 6 个 | **一个模型覆盖 6 个评测设置**，正式提交用 |

**联合模式的三处技术处理**（这也是 single 与 joint 能共用一套代码的原因）：

1. **统一输出维度**：四个子集都用 2 列输出（复合词：修饰语/中心语；小品词动词：只有第 1 列有标签，
   第 2 列用 `loss_mask` 屏蔽）。模型结构、损失、指标、绘图因此完全不用判断子集类型。
2. **分组键带子集**：`group_index` 编的是 **(子集, 目标)** 而不是单纯的目标。
   指标 2 的组内 std 必须按这个分组算，否则不同子集里同名目标会被串成一组。
3. **切分在子集内部做**：每个子集各自按「目标级互斥 + 先分箱后分层抽样」切分，
   再把四个子集的 train/val 拼起来。这样每个子集的验证集难度分布都保持正确。

**想少跑几次**：直接注释掉 `RUN_PLAN` 里对应的行，其余代码一行不用动。

### 云端跑法（推荐命令）

```bash
# 1) 拉代码与数据（数据放在仓库的 train/ 下）
git clone <your-repo> ComPartMENT && cd ComPartMENT

# 2) 环境变量（都可省略，notebook 会自动探测）
export COMPARTMENT_TRAIN_DIR=/kaggle/input/datasets/kintsugi0v0/compartment-data/train
export COMPARTMENT_OUTPUT=/kaggle/working/outputs
export COMPARTMENT_HF_MODE=auto        # auto=先探测官方源，不通换镜像
export TOKENIZERS_PARALLELISM=false

# 3) 依赖
pip install "transformers>=5" "huggingface-hub<2.0" "datasets<4" "pyarrow<18" \
            scikit-learn pandas matplotlib seaborn scipy jupyter nbconvert

# 4) 先用 30 秒跑通整条链路（2 层随机主干，不下权重）
#    把 Step1 的 SMOKE 改成 True
jupyter nbconvert --to notebook --execute \
  "ComPartMENT-CoMWE基线(Qwen2.5-1.5B-Instruct-回归)/*.ipynb" \
  --ExecutePreprocessor.timeout=1800

# 5) 正式跑（把 Step1 的 SMOKE 改回 False）
nohup jupyter nbconvert --to notebook --execute \
  "ComPartMENT-CoMWE基线(Qwen2.5-1.5B-Instruct-回归)/*.ipynb" \
  --ExecutePreprocessor.timeout=7200 > train.log 2>&1 &
```

**数据路径不用改代码**：notebook 会按「环境变量 `COMPARTMENT_TRAIN_DIR` → Kaggle 挂载目录
`/kaggle/input/datasets/kintsugi0v0/compartment-data/train` → 仓库内 `train/`」的顺序自动探测，
命中即用。Kaggle 上直接跑就是挂载目录，不用改任何一行。

**云端唯一必须改的地方**（Step1）：

| 位置 | 改什么 | 为什么 |
|---|---|---|
| `SMOKE` | `False` | `True` 是 30 秒冒烟模式，不产出可用模型 |
| `HF_MODE` | 一般不用改 | `auto` 会先探测官方源；海外云端直通用它最稳 |

### 显存不够时的降级顺序（代价从小到大）

Step8 的**训练档位机制**已经把这套顺序自动化了（见上文的档位表），
这里只列「为什么是这个顺序」：

1. **降 `per_device_train_batch_size`，用 `gradient_accumulation_steps` 把有效 batch 补回去**
   —— 数学等价、几乎无损，第一选择（显存探测会自动做这件事）
   —— ⚠️ 但只省**激活**显存，救不了「权重+梯度+优化器状态」这块固定开销
2. **开 `gradient_checkpointing`**（记得同时 `model.config.use_cache = False`）
   —— 激活降到约 1/N，慢约 30%
3. **换优化器**（`adafactor` 把 m/v 从 8 B/param 压到 ≈0.2；`adamw_bnb_8bit` 压到 2）
   —— 唯一能显著降低**固定开销**的办法
4. **关掉混合精度**（AMP 的 fp16 权重副本占 2 B/param，Qwen-1.5B 就是差这 3.1 GiB）
   —— 代价是纯 fp32 训练慢一些
5. **只训顶部若干层**（`apply_fit_scope`）—— 会冻结底层与 embedding，是**方法层面的改变**，
   两条基线必须同步，且必须记进口径清单
6. **降 `MAX_LENGTH`** —— 最有效，但要改全局配置并**从头重跑**（分词在 Step3 已固化）
7. **换更小的同族模型** —— `microsoft/deberta-v3-base` / `Qwen2.5-0.5B-Instruct`，
   管线一行不用改

### 三个判断「实验有没有做对」的指标

1. **梯度流验证必须通过**：回归头与主干的梯度绝对值和都 > 0，且 loss 不是 NaN。
   否则模型可能训了十几分钟而参数纹丝不动。
2. **指标 2 不能是 `nan`**：组内预测 std 恒为 0 会让相关系数未定义。
   图 C 上如果点全挤在水平线上，就说明模型没有真正对语境敏感。
3. **训练后必须显著优于随机初始化**（参考对照区）：「训练后模型」的指标 1 要明显高于
   「随机初始化」，否则模型没有学到任何东西（此时优先查：标签列是否对齐、
   输入模板是否恒定、学习率是否过大）。

### 后续可以做什么（对应本项目的建模决策）

| 方向 | 做法 | 开关位置 |
|---|---|---|
| 加组内排序损失 | 提升指标 2，同时对标签噪声更稳健 | `LAMBDA_RANK` |
| 加组内 std 匹配损失 | 直接对着指标 2 的金标准优化 | `MU_STD` |
| 换模型规模 | `deberta-v2-xlarge` → `deberta-v3-xlarge`；`Qwen2.5-1.5B` → `7B` | `MODEL_REGISTRY` |
| 减少运行次数 | 只跑 joint 一次即可覆盖全部 6 个评测设置 | `RUN_PLAN` |
| 改进定位 | 德语复合词必须预切分（不能字符串切片还原），切分版与未切分版 ensemble | Step3 的 `locate_target` |
| 帕累托前沿 | 把输出缩放系数当旋钮，在 dev 上画「指标1 ρ vs 指标2 ρ」的前沿，而不是用单一指标选点 | Step9 之后 |

### 复现命令

```bash
cd ComPartMENT
pip install "transformers>=5" "huggingface-hub<2.0" "datasets<4" "pyarrow<18" scikit-learn pandas matplotlib seaborn scipy
jupyter nbconvert --to notebook --execute \
  "ComPartMENT-CoMWE基线(Qwen2.5-1.5B-Instruct-回归)/*.ipynb"
```